# 🧱 Unit 4 · Module 1 · Notebook 4: How a Context Fails and How to Fix It

### Build the context of a bank support bot, measure what it costs, break it in five ways and fix each one

**What this notebook is about.** A model answers from what it reads in one call: the instructions, the documents, the conversation so far and the question. All of that together is called the **context**. A model can have every fact it needs in its context and still answer wrongly, because of what else is in there: a wrong number from an earlier reply, sixty pages around the one paragraph that matters, forty tools where three are needed, two documents that disagree. In this notebook you write the function that builds the context for a bank's support bot, you measure what that context costs, and then you break it on purpose in five ways. Each time you run the bot while the context is broken, you fix the context with a few lines of code, and you run it again. For most of the problems you count the right answers before and after. For the others you measure what really changed: the tokens you pay for, or the quotes you can check. At the end you have one `build_context()` function with every fix in it, and a table that shows what each fix changed.

**The example we use.** A bank's support bot answers customers' questions about the fees of personal accounts. You get the bank's fee policy (three pages, 12 sections) and **20 questions about it with known answers**, so code can check every reply. One function, `score()`, asks the bot all 20 questions with the context you built and counts the right answers. It is the ruler for the whole notebook.

| Part | What you do with the context | Exercises |
|---|---|---|
| **Build and pay** | assemble it, count its tokens, watch it grow in a conversation, order it for the cache | 1, 2, 3 |
| **Break and fix** | poisoning, distraction, confusion, clash, ignored context: each one built, run and fixed | 4, 5, 6, 7, 8 |
| **Put it together** | every fix in one function, against a context with all five problems | 9 |

**Before you start:** you need an OpenRouter key in Colab's Secrets panel (the Setup section says how), and four files that you upload once (the section after Setup lists them).

---

### 🗺️ What You'll Do

1. **Assemble the context** from four parts, count the tokens of each part, and measure the baseline on the 20 questions. (Exercise 1)
2. **Hold a ten-turn conversation** and plot how the input tokens and the cost grow with every turn. (Exercise 2)
3. **Order the context for the prompt cache**: the current time at the top, then at the end, and read the cached tokens of each call. (Exercise 3)
4. **Poison the conversation history** with a wrong daily limit, count the answers that repeat it, then check every reply with code before it is saved. (Exercise 4)
5. **Replace two policy sections with the bank's full terms and conditions**, measure what that costs per question, then select the matching sections and ask for quotes. (Exercise 5)
6. **Offer the bot 40 tools where 3 are needed**, count the calls to a wrong tool, then pass only the tools that match the question. (Exercise 6)
7. **Put an outdated FAQ page next to the current policy**, count which number comes back, and try two fixes. (Exercise 7)
8. **Ask five questions whose answer is unusual for a bank** with no document, with the policy, and with an instruction that makes every answer checkable. (Exercise 8)
9. **Put every fix into `build_context()`** and run it against a context that has all five problems at once. (Exercise 9)

After every exercise you read **the bill**: how many calls, how many tokens went in, and what they cost.

---

### 🧭 How to Use This Notebook

- **Task** notebook: fill the blanks (`___`), guided by the numbered `# TODO` comments. **Answer** notebook: the same, already filled in.
- Run the cells from top to bottom: later exercises use what earlier ones built. Every exercise ends with ✅ / ❌ checks on the things your code controls. A 🤔 is not an error: it marks a result that depends on what the model answered in your run, with a note on how to read it.
- The notebook uses two models: a larger one for most exercises, and a small one for Exercises 4, 5, 8 and 9, where the larger one resists the failure so well that there would be nothing to measure. Even so, a failure may not appear in your run. That is a real result, and the 🤔 line says how to read it. The fix is still worth knowing: it turns "usually right" into something your code guarantees.
- The whole notebook makes about 420 calls and takes about 3 to 5 minutes to run. Most calls are sent up to 8 at a time.
- Total cost of this notebook: about **10 to 15 US cents**. About half of it comes from three runs that put a document of about 60 pages into every call (Exercises 5 and 9). The notebook prints the running total after every exercise.
- **Files to upload:** `bank_fee_policy.md`, `fee_policy_questions.json`, `bank_terms_and_conditions.md` and `bank_terms_wrong_sources.json`. You find them in the same place as this notebook.

💡 *Where the data comes from:* all four files were made for this course, and no real bank or customer is in them. The fee policy was written by hand. The 20 questions were written from it, and each one carries a pattern that code uses to check a reply. The terms and conditions were built by a script with a fixed starting number: it spreads the 12 sections of the fee policy among many other sections, adds look-alike sections with different numbers (business accounts, credit cards, an older fee schedule), and writes down which number stands in which section.

# 🔑 Setup: Your OpenRouter Key and the Shared Helpers

**The key.** This notebook calls a hosted model through **OpenRouter**, one website that passes your request to the model you choose. It needs a secret named `OPENROUTER_API_KEY` in Colab's **🔑 Secrets** panel (left sidebar), with **Notebook access** switched on. If you do not have a key yet: create one at https://openrouter.ai/settings/keys, add a few dollars of credit, and save it as that secret. The steps are shown in detail in Unit 2 · Module 1 · Notebook 1: Your First Call.

**The two models.** Most exercises use `anthropic/claude-haiku-5.5`: 0.10 USD per million tokens in, 0.50 USD per million tokens out (checked on 2026-11-14; the cell looks the prices up again). A **token** is roughly a piece of a word. You pay for the tokens the model reads ("in") and for the tokens it writes ("out").

Three things about this first model matter in this notebook:

- **It thinks before it answers.** It first writes notes to itself, then the answer, and you pay for those notes as output tokens. A setting called **reasoning effort** says how much it may think. We set it to `low` for every call.
- **It does not accept `temperature`**, the setting that makes answers more or less random. So no call here sets it, and the same question can get slightly different answers in two runs.
- **Its price is five times higher for a request above 100,000 input tokens.** Every request in this notebook stays far below that.

Exercises 4, 5, 8 and 9 use a second, smaller model: `mistralai/ministral-8b-2512` (0.15 USD per million tokens, in and out, checked on 2026-11-14). A small, fast model like this is a realistic choice for a support bot with many customers. It also gives in to a bad context more easily than the larger model, which corrected or avoided most of the planted failures in our runs, so on the small model you can see the failures and measure the fixes. It does not think before it answers, and it accepts `temperature`, which we set to 0, so that its answers change as little as possible from run to run.

**The setup we recommend.** The results in this notebook were measured with these models and these settings, so run it this way first: it is where each technique's effect is easiest to see. How much a technique adds depends on the task and the model. On a harder task or a smaller model the difference is usually larger; on an easy task with a strong model it can be small.

**This cell also creates the helpers every exercise uses:**

- **`call_model(messages, tools=None)`:** sends one request and returns the reply with everything we measure: tokens in and out, tokens read from the cache, cost and seconds. If a call fails it tries once more, then writes the error down and moves on.
- **`ask_bot(context)`:** asks the bot one question with a context you built. If the bot asks for a tool, this function runs the tool, sends the result back and asks again, up to three times.
- **`score(build_context)`:** asks the bot all 20 policy questions, each with the context that your `build_context(question)` returns, and counts the right answers. It also adds up the input tokens and the cost. With `model=SMALL_MODEL` it runs on the small model.
- **`count_tokens(text)`:** an estimate of how many tokens a text has, for looking at the parts of a context before you send it.
- **`rank_by_word_overlap(question, texts)`:** sorts texts by how many meaningful words they share with a question. You use it to pick policy sections and tools.
- **`record_result(...)`:** writes one row into `RESULTS`, the table you complete in Exercise 9: what was measured for a problem, before and after the fix.

The cell also holds three things the bank gives you: the bot's **system prompt** (its standing instructions), the bank's **catalog of 40 tools** (Exercise 6 explains what a tool is), and an **old FAQ page** from the help center (Exercise 7).

In [ ]:
# ==========================================================================
# 🔧 Setup: run this cell once, before any exercise
# ==========================================================================

# ==========================================================================
# Step 1: Install the libraries this notebook uses
# ==========================================================================
# Colab usually has them already; this line only adds what is missing.
# - openai: sends requests to OpenRouter (OpenRouter accepts the same request format as OpenAI's API)
# - tiktoken: cuts text into tokens, so we can estimate the size of each part of a context
%pip install -q openai tiktoken

# ==========================================================================
# Step 2: Import the tools
# ==========================================================================
import json                                         # json: turns Python dictionaries into text and back
import math                                         # math: log() for the word weights in rank_by_word_overlap
import os                                           # os: checks whether a file exists
import re                                           # re: finds patterns in text, like "$3" or "45 days"
import textwrap                                     # textwrap: breaks long text into readable lines
import time                                         # time: measures how long each call takes
from concurrent.futures import ThreadPoolExecutor   # ThreadPoolExecutor: runs several calls at the same time
from datetime import datetime                       # datetime: the current time, for Exercise 3
import matplotlib.pyplot as plt                     # matplotlib: charts
import pandas as pd                                 # pandas: tables of results
import requests                                     # requests: sends requests over the internet
import tiktoken                                     # tiktoken: cuts text into tokens
from openai import OpenAI                           # OpenAI: the client we use to call models through OpenRouter
import os
# --- Local (VS Code) stand-ins for google.colab ---------------------------
from pathlib import Path
from dotenv import load_dotenv                      # load_dotenv: reads keys from the .env file next to this notebook
load_dotenv("D:/_Repos/_AIA/.env")

class userdata:  # replaces google.colab.userdata: reads the key from .env or an environment variable
    @staticmethod
    def get(name):
        value = os.environ.get(name)
        if not value:
            raise KeyError(f"{name} is not set in .env or the environment")
        return value

class files:  # replaces google.colab.files: VS Code has no upload picker, so files must sit in this folder
    @staticmethod
    def upload():
        print(f"   (local run) No upload picker here. Copy the file into: {Path.cwd()}")
        print("   Then run this cell again.")
        return {}
# ---------------------------------------------------------------------------

# ==========================================================================
# Step 3: Read your OpenRouter key from Colab Secrets
# ==========================================================================
# If the secret is missing, we stop with a message that says how to fix it.
try:
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    raise RuntimeError(
        "❌ No key found. Add OPENROUTER_API_KEY=<your key> to the .env file "
        "next to this notebook, then run this cell again."
    ) from None

# ==========================================================================
# Step 4: Choose the model and look up its prices
# ==========================================================================
# OpenRouter publishes prices per single token; we multiply by 1,000,000 to get USD per million tokens.
# Four prices matter here: tokens in, tokens out, and two prices for the prompt cache (Exercise 3 explains them).
MODEL = "anthropic/claude-haiku-5.5"
PRICE_IN, PRICE_OUT = 0.10, 0.50                      # fallback prices (checked on 2026-11-14), if the lookup fails
PRICE_CACHE_READ, PRICE_CACHE_WRITE = 0.01, 0.125     # reading tokens from the cache, writing tokens into it
try:
    for model_info in requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]:
        if model_info["id"] == MODEL:
            pricing = model_info["pricing"]
            PRICE_IN = float(pricing["prompt"]) * 1_000_000        # price of tokens the model reads
            PRICE_OUT = float(pricing["completion"]) * 1_000_000   # price of tokens the model writes
            PRICE_CACHE_READ = float(pricing.get("input_cache_read", PRICE_CACHE_READ / 1_000_000)) * 1_000_000
            PRICE_CACHE_WRITE = float(pricing.get("input_cache_write", PRICE_CACHE_WRITE / 1_000_000)) * 1_000_000
except Exception:
    print("⚠️ Could not look up the prices; using the fallback prices.")

# Reasoning effort: how much the model may think before it answers ("low" = a little).
# Every call to MODEL uses the same effort, so differences come from the context, not from this setting.
EFFORT = "low"

# The second, smaller model, used in Exercises 4, 5, 8 and 9. It does not think before it answers, and it accepts
# temperature. We call it at temperature 0: always take the most likely next token, so answers are as repeatable as possible.
SMALL_MODEL = "mistralai/ministral-8b-2512"
SMALL_PRICE_IN, SMALL_PRICE_OUT = 0.15, 0.15   # fallback prices (checked on 2026-11-14), if the lookup fails
try:
    for model_info in requests.get("https://openrouter.ai/api/v1/models", timeout=30).json()["data"]:
        if model_info["id"] == SMALL_MODEL:
            SMALL_PRICE_IN = float(model_info["pricing"]["prompt"]) * 1_000_000
            SMALL_PRICE_OUT = float(model_info["pricing"]["completion"]) * 1_000_000
except Exception:
    print("⚠️ Could not look up the price of the small model; using the fallback price.")

# ==========================================================================
# Step 5: Create the client that sends our requests to OpenRouter
# ==========================================================================
# max_retries=0: the library does not retry by itself. call_model below retries once.
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY, max_retries=0)

# ==========================================================================
# Step 6: One function that makes a call and measures everything about it
# ==========================================================================
# Every call in this notebook goes through call_model. It records the reply, the tokens, the cost and the seconds.
ALL_CALLS = []              # every call this notebook makes is written down here (used for the running total)
MAX_TOKENS = 3000           # the most tokens the model may write, thinking included: room enough that no answer is cut off
CALL_TIMEOUT_SECONDS = 180  # give up on a call after 3 minutes


def call_model(messages, tools=None, max_tokens=MAX_TOKENS, model=None):
    """Send one request to a model. Return a dictionary with the reply and what we measure. Never raises an error.

    messages: the context, as a list of {"role": ..., "content": ...} dictionaries.
    tools: an optional list of tool descriptions the model may ask our code to run (Exercise 6).
    model: MODEL unless you pass SMALL_MODEL.
    """
    model = model or MODEL
    request = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "timeout": CALL_TIMEOUT_SECONDS,
    }
    if model == SMALL_MODEL:
        request["temperature"] = 0   # the small model does not think; it accepts temperature
    else:
        # extra_body carries settings that only OpenRouter knows. "reasoning" is its word for thinking.
        request["extra_body"] = {"reasoning": {"effort": EFFORT}}
    if tools:
        request["tools"] = tools
    record = {"text": "", "message": None, "finish_reason": None, "tokens_in": 0, "tokens_out": 0,
              "cached_tokens": 0, "cache_write_tokens": 0, "cost": 0.0, "seconds": 0.0, "error": None}
    for attempt in (1, 2):   # one try, and one retry if the first try fails
        start = time.perf_counter()   # perf_counter: a precise clock for measuring how long something takes
        try:
            # **request hands every key of the dictionary to the function as a named setting
            response = client.chat.completions.create(**request)
            record["seconds"] = time.perf_counter() - start
            if not getattr(response, "choices", None):   # OpenRouter answered, but without a reply from the model
                raise RuntimeError(f"no reply in the response: {getattr(response, 'error', None)}")
            choice = response.choices[0]
            record["message"] = choice.message                # kept whole: ask_bot sends it back when a tool was called
            record["text"] = (choice.message.content or "").strip()
            record["finish_reason"] = choice.finish_reason    # "stop" = finished; "length" = cut off at max_tokens
            usage = response.usage.model_dump()               # the token counts of this call, as a dictionary
            record["tokens_in"] = usage.get("prompt_tokens") or 0        # tokens the model read (cached ones included)
            record["tokens_out"] = usage.get("completion_tokens") or 0   # tokens the model wrote, thinking included
            # The prompt cache (Exercise 3): how many of the input tokens were read from the cache at a tenth of
            # the price, and how many were written into the cache for later calls.
            details = usage.get("prompt_tokens_details") or {}
            record["cached_tokens"] = details.get("cached_tokens") or 0
            record["cache_write_tokens"] = details.get("cache_write_tokens") or 0
            # OpenRouter reports what it charged for the call in usage["cost"] (in credits; one credit is one US dollar).
            # If that field is missing, we work the cost out from the token counts and the prices per million tokens.
            billed = usage.get("cost")
            if billed is None and model == SMALL_MODEL:
                billed = (record["tokens_in"] * SMALL_PRICE_IN + record["tokens_out"] * SMALL_PRICE_OUT) / 1_000_000
            elif billed is None:
                full_price_tokens = record["tokens_in"] - record["cached_tokens"] - record["cache_write_tokens"]
                billed = (full_price_tokens * PRICE_IN + record["cached_tokens"] * PRICE_CACHE_READ
                          + record["cache_write_tokens"] * PRICE_CACHE_WRITE + record["tokens_out"] * PRICE_OUT) / 1_000_000
            record["cost"] = float(billed)
            record["error"] = None
            break   # the call worked: no retry needed
        except Exception as error:
            record["seconds"] = time.perf_counter() - start
            record["error"] = f"{type(error).__name__}: {str(error)[:200]}"   # [:200] keeps the message short
            if attempt == 1:
                time.sleep(2)   # wait two seconds, then try once more
    ALL_CALLS.append(record)
    return record


# ==========================================================================
# Step 7: Run many calls at the same time
# ==========================================================================
# A call spends most of its time waiting for the model. While one call waits, others can be sent.
def run_in_parallel(function, items, workers=8):
    """Run function(item) for every item, up to `workers` at the same time. Return the results in the same order."""
    with ThreadPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(function, items))


# ==========================================================================
# Step 8: The bill for a list of calls
# ==========================================================================
def show_bill(calls, title):
    """Print the number of calls, the tokens and the cost of a list of calls, and the total spent so far."""
    cost = sum(call["cost"] for call in calls)
    tokens_in = sum(call["tokens_in"] for call in calls)
    tokens_out = sum(call["tokens_out"] for call in calls)
    cached = sum(call["cached_tokens"] for call in calls)
    print(f"🧾 Bill for {title}: {len(calls)} calls · {tokens_in:,} tokens in ({cached:,} of them read from the cache) · "
          f"{tokens_out:,} tokens out · ${cost:.4f}")
    failed = [call for call in calls if call["error"]]
    if failed:
        print(f"   ⚠️ {len(failed)} of these calls failed twice and have no reply. First error: {failed[0]['error']}")
    print(f"   Spent so far in this notebook: ${sum(call['cost'] for call in ALL_CALLS):.4f} ({len(ALL_CALLS)} calls)")
    return cost


# ==========================================================================
# Step 9: Estimate the tokens of a text
# ==========================================================================
# tiktoken cuts text the way OpenAI's models do. Our model cuts text a little differently and usually gets
# somewhat MORE tokens, so this is an estimate. The exact number of a request is in the bill, after the call.
TOKEN_ENCODER = tiktoken.get_encoding("o200k_base")


def count_tokens(text):
    """An estimate of how many tokens a text has."""
    return len(TOKEN_ENCODER.encode(text))


# ==========================================================================
# Step 10: Sort texts by how well they match a question
# ==========================================================================
# A simple way to find the texts that belong to a question: count the meaningful words they share.
# A word that appears in few of the texts ("overdraft") counts more than a word that appears in most ("fee").
# Real systems often use a search engine or embeddings for this step. Word overlap is enough here, and you can read it.
COMMON_WORDS = set("a an the of to in on at for is are was be by or and with what how does do many much one it its that "
                   "this which when who from as not no any can has have per there than then if we our".split())


def meaningful_words(text):
    """The set of words in a text, lower case, without very common words; a final 's' is cut off (fees -> fee)."""
    found = set()
    for word in re.findall(r"[a-z]+", text.lower()):   # every run of letters is one word
        if word in COMMON_WORDS or len(word) < 3:
            continue
        found.add(word[:-1] if word.endswith("s") and len(word) > 3 else word)
    return found


def rank_by_word_overlap(question, texts):
    """Return the positions of `texts`, best match with the question first."""
    word_sets = [meaningful_words(text) for text in texts]
    scores = []
    for words in word_sets:
        score_of_text = 0.0
        for word in sorted(meaningful_words(question)):   # sorted: the same order, and so the same sum, in every run
            if word in words:
                texts_with_word = sum(word in other for other in word_sets)   # True counts as 1
                score_of_text += math.log(1 + len(texts) / texts_with_word)   # a rare word counts more
        scores.append(round(score_of_text, 6))
    # sorted() is stable: texts with the same score keep their original order
    return sorted(range(len(texts)), key=lambda position: -scores[position])


# ==========================================================================
# Step 11: What the bank gives you: the bot's system prompt and two earlier turns of a conversation
# ==========================================================================
# The system prompt is the bot's standing instructions: who it is and how it answers. It is the same in every call.
SYSTEM_PROMPT = (
    "You are the support assistant of a bank. You answer customers' questions about the fees and limits of "
    "personal accounts. There are three plans: Standard, Plus and Premium.\n"
    "- Answer in one or two short sentences, in plain words.\n"
    "- State the exact amount, number or time limit the customer asked about.\n"
    "- Be polite and direct. Do not add offers or advice the customer did not ask for.\n"
    "- If you cannot answer, say so and offer to connect the customer with a support agent."
)

# The conversation so far: two earlier turns, each a pair (what the customer wrote, what the bot replied).
EARLIER_TURNS = [
    ("Hi, I have a few questions about my account. First: does the Standard plan have a monthly fee?",
     "No. The Standard plan has no monthly fee."),
    ("Good. And do I pay anything for electronic statements?",
     "No. Electronic statements are free."),
]


def turns_as_messages(turns):
    """Turn a list of (customer text, bot reply) pairs into chat messages: user, assistant, user, assistant, ..."""
    messages = []
    for customer_text, bot_reply in turns:
        messages.append({"role": "user", "content": customer_text})        # "user": the customer
        messages.append({"role": "assistant", "content": bot_reply})       # "assistant": the bot's earlier reply
    return messages


# ==========================================================================
# Step 12: What the bank gives you: an old FAQ page from the help center (Exercise 7)
# ==========================================================================
# The page was written in 2024. Four of its numbers are no longer true: the current fee policy changed them.
OLD_FAQ_TEXT = """Fees: Frequently Asked Questions

How much does it cost to use another bank's ATM?
Withdrawals at our own ATMs are free. On the Standard plan, each withdrawal at another bank's ATM inside the country costs $2.50.

What does a replacement card cost?
Your first replacement card in a calendar year is free. Every further replacement card in the same year costs $6.

I see a charge I do not recognise. What can I do?
You can dispute the charge within 30 days of the date of the statement on which it first appears. Opening a dispute is free.

Can I still get paper statements?
Yes. Electronic statements are free. A paper statement sent by post costs $3 per month.

How do I avoid the monthly fee of the Plus plan?
Keep an average daily balance of $1,500 or more, and the monthly fee is waived for that month.
"""

# ==========================================================================
# Step 13: What the bank gives you: the catalog of 40 tools (Exercise 6)
# ==========================================================================
# A tool is a function of the bank's systems that the bot may ask our code to run. The model only sees each
# tool's name, its description and its inputs. Each row: (name, description, {input name: what it is}).
TOOL_CATALOG = [
    # Three tools return the current fee policy for personal accounts, one part each. The 20 questions need only these.
    ("get_account_fee_policy",
     "Current fee policy for personal accounts, part 1: the monthly fee of each plan (Standard, Plus, Premium) and the "
     "average daily balance that waives it, inactive accounts and the inactivity fee, closing an account and the early "
     "closure fee, courtesy refunds of a fee.", {}),
    ("get_card_and_atm_fee_policy",
     "Current fee policy for personal accounts, part 2: cash withdrawals at ATMs and the daily ATM withdrawal limit, "
     "card purchases in a foreign currency, replacement cards and card delivery, disputing a card charge (how many "
     "days a customer has, counted from the statement date) and the provisional credit. Includes the fee at another bank's ATM inside the country and abroad, and the free "
     "withdrawals per month on the Plus plan.", {}),
    ("get_payment_fee_policy",
     "Current fee policy for personal accounts, part 3: domestic and instant transfers, outgoing and incoming "
     "international wires and the currency conversion fee, overdrafts and overdraft fees (the fee per item and the largest number of fees per day), paper statements, cheques, "
     "stop payments, cash deposits.", {}),
    # Six tools return information that looks similar but is about another product, another country or an older time.
    ("get_fee_schedule",
     "The Bank's fee schedule by product: monthly fees, ATM withdrawals, card purchases, transfers and overdrafts.",
     {"product": "personal, business or credit_card"}),
    ("search_help_center",
     "Search the help center articles that customers read on the website: guides and frequently asked questions "
     "about fees, cards, ATMs and statements.", {"query": "what to search for"}),
    ("get_business_account_fees",
     "Fees for business accounts (Business Standard, Business Plus): monthly fee, ATM withdrawals, wires, overdrafts.", {}),
    ("get_credit_card_fees",
     "Fees for credit cards: annual fee, cash advances at ATMs, purchases in a foreign currency, late payment.", {}),
    ("get_savings_account_terms",
     "Terms of savings accounts: interest, free withdrawals per month, the fee for extra withdrawals.", {}),
    ("get_canada_branch_fees",
     "Fees for personal accounts held at branches in Canada, in Canadian dollars.", {}),
    # Eleven tools read one customer's data.
    ("get_account_balance", "The current balance of a customer's account.", {"customer_id": "the customer's id"}),
    ("get_recent_transactions", "The last 30 days of payments, withdrawals and transfers on a customer's account.",
     {"customer_id": "the customer's id"}),
    ("get_card_status", "Whether a card is active, frozen or expired.", {"card_id": "the card's id"}),
    ("get_customer_plan", "Which plan a customer is on (Standard, Plus or Premium) and since when.",
     {"customer_id": "the customer's id"}),
    ("get_fees_charged", "The fees that were charged to a customer's account in the last 90 days.",
     {"customer_id": "the customer's id"}),
    ("get_overdraft_status", "Whether a customer's account is overdrawn right now, and by how much.",
     {"customer_id": "the customer's id"}),
    ("get_dispute_status", "The state of a dispute a customer opened earlier.", {"dispute_id": "the dispute's id"}),
    ("get_wire_status", "Where an international wire is right now.", {"wire_id": "the wire's id"}),
    ("get_cheque_status", "Whether a cheque a customer paid in has cleared.", {"cheque_number": "the cheque's number"}),
    ("get_statement", "One monthly statement of a customer's account.",
     {"customer_id": "the customer's id", "month": "the month, as YYYY-MM"}),
    ("get_atm_withdrawals_today", "How much a customer has withdrawn at ATMs today.", {"customer_id": "the customer's id"}),
    # Twelve tools change something.
    ("freeze_card", "Freeze a card so that it cannot be used.", {"card_id": "the card's id"}),
    ("unfreeze_card", "Make a frozen card usable again.", {"card_id": "the card's id"}),
    ("order_replacement_card", "Order a replacement card for a customer.", {"card_id": "the card to replace"}),
    ("open_dispute", "Open a dispute for one charge.", {"transaction_id": "the charge to dispute"}),
    ("refund_fee", "Refund one fee to a customer as a courtesy.", {"fee_id": "the fee to refund"}),
    ("change_plan", "Move a customer to another plan from next month.",
     {"customer_id": "the customer's id", "new_plan": "Standard, Plus or Premium"}),
    ("close_account", "Close a customer's account.", {"customer_id": "the customer's id"}),
    ("send_wire", "Send an international wire from a customer's account.",
     {"customer_id": "the customer's id", "amount": "the amount in US dollars"}),
    ("cancel_wire", "Cancel a wire that has not left the bank yet.", {"wire_id": "the wire's id"}),
    ("stop_payment", "Place a stop payment on a cheque or a recurring payment.", {"payment_id": "the payment to stop"}),
    ("set_temporary_atm_limit", "Raise a customer's daily ATM limit for up to 7 days.",
     {"customer_id": "the customer's id", "limit": "the new limit in US dollars"}),
    ("order_paper_statement", "Switch a customer to paper statements by post.", {"customer_id": "the customer's id"}),
    # Eight more tools for other jobs.
    ("find_nearest_atm", "The ATMs of the Bank that are closest to an address.", {"address": "street and city"}),
    ("get_exchange_rate", "Today's exchange rate between the US dollar and another currency.",
     {"currency": "a currency code such as EUR"}),
    ("get_branch_opening_hours", "The opening hours of one branch.", {"branch": "the name of the branch"}),
    ("get_interest_rates", "Today's interest rates for savings accounts and loans.", {}),
    ("get_loan_offers", "The personal loans a customer could apply for.", {"customer_id": "the customer's id"}),
    ("escalate_to_human", "Hand the conversation over to a support agent.", {"reason": "why the bot cannot continue"}),
    ("send_secure_message", "Send a message to the customer's inbox in the app.", {"message": "the text to send"}),
    ("book_branch_appointment", "Book an appointment at a branch.", {"date": "the day, as YYYY-MM-DD"}),
]


def as_tool(name, description, inputs):
    """One catalog row in the form the API expects: a name, a description, and the inputs as a JSON schema."""
    return {"type": "function", "function": {
        "name": name,
        "description": description,
        "parameters": {"type": "object",
                       "properties": {key: {"type": "string", "description": text} for key, text in inputs.items()},
                       "required": list(inputs)},
    }}


ALL_TOOLS = [as_tool(*row) for row in TOOL_CATALOG]   # *row hands the three parts of a row to as_tool

# Which sections of the fee policy each of the three policy tools returns
POLICY_TOOL_SECTIONS = {
    "get_account_fee_policy": [2, 10, 11, 12],
    "get_card_and_atm_fee_policy": [3, 4, 7, 8],
    "get_payment_fee_policy": [5, 6, 9],
}

# What the six look-alike tools return: real-looking information with numbers that are wrong for our questions
LOOK_ALIKE_RESULTS = {
    "get_fee_schedule": (
        "Fee schedule, personal accounts (schedule of 1 January 2024). Another bank's ATM: $2.50 per withdrawal on "
        "Standard; first 2 per month free on Plus. Daily ATM limit: $400. Plus plan: $5 per month, waived from an "
        "average daily balance of $1,000. Foreign currency purchases on Standard: 2.5%. Outgoing international wire: "
        "$22. Instant transfer: 1%, at most $7.50. Overdraft fee: $28 per item, at most 4 per day. Second replacement "
        "card: $6. Express delivery: $15. Paper statement: $3 per month. Returned cheque: $10. Stop payment: $12.50. "
        "Inactive after 24 months, then $3 per month. Disputes within 30 days; provisional credit within 12 business "
        "days. Early closure (within 90 days): $20."),
    "search_help_center": OLD_FAQ_TEXT,
    "get_business_account_fees": (
        "Business accounts. Business Standard: $22 per month. Another bank's ATM: $4.50 per withdrawal. "
        "Daily ATM limit: $1,000. Outgoing international wire: $32. Overdraft fee: $34 per item."),
    "get_credit_card_fees": (
        "Credit cards. Cash advance at an ATM: $10. Purchases in a foreign currency: 2.7%. Late payment: $29. "
        "Replacement card: $11. Disputes: within 75 days of the statement date."),
    "get_savings_account_terms": (
        "Savings accounts. 6 free withdrawals per month, then $2 per withdrawal. No monthly fee. "
        "Inactive after 36 months without a transaction."),
    "get_canada_branch_fees": (
        "Personal accounts at branches in Canada (amounts in Canadian dollars). Another bank's ATM: $4 per "
        "withdrawal on Standard. Daily ATM limit: $600. Plus plan: $7 per month. Paper statement: $5 per month."),
}


def run_tool(name, arguments):
    """Run one tool and return its result as text. This is ordinary code: the model only asked for it."""
    if name in POLICY_TOOL_SECTIONS:
        # POLICY_SECTIONS is loaded in Exercise 1: the 12 sections of the fee policy
        wanted = POLICY_TOOL_SECTIONS[name]
        return "\n\n".join(section["text"] for section in POLICY_SECTIONS if section["number"] in wanted)
    if name in LOOK_ALIKE_RESULTS:
        return LOOK_ALIKE_RESULTS[name]
    if name in [row[0] for row in TOOL_CATALOG]:
        return "This is a practice system: no customer is logged in, so this tool has nothing to return."
    return f"Error: there is no tool named {name}."


def right_tool_for(question):
    """The name of the one policy tool that holds the answer to a question (looked up by the question's section)."""
    return next(name for name, sections in POLICY_TOOL_SECTIONS.items() if question["section"] in sections)


# ==========================================================================
# Step 14: Ask the bot one question with a context you built
# ==========================================================================
# A context is either a list of messages, or a dictionary {"messages": [...], "tools": [...]} when the bot
# also gets tools. If the model asks for a tool, we run it, add the result to the messages and call the model again.
MAX_MODEL_CALLS = 3   # at most three calls per question: enough for two rounds of tools and the final answer


def ask_bot(context, model=None):
    """Ask the bot with this context. Return its answer, the names of the tools it called, and the bill."""
    if isinstance(context, dict):
        messages, tools = list(context["messages"]), context.get("tools") or None
    else:
        messages, tools = list(context), None   # list(...) makes a copy, so the context you built is not changed
    answer = {"text": "", "tools_called": [], "calls": [], "tokens_in": 0, "cached_tokens": 0, "cost": 0.0, "error": None}

    for _ in range(MAX_MODEL_CALLS):
        call = call_model(messages, tools=tools, model=model)
        answer["calls"].append(call)
        answer["tokens_in"] += call["tokens_in"]
        answer["cached_tokens"] += call["cached_tokens"]
        answer["cost"] += call["cost"]
        if call["error"]:
            answer["error"] = call["error"]
            break

        # Did the model ask for a tool? If not, this is its reply to the customer.
        tool_calls = getattr(call["message"], "tool_calls", None) or []   # None becomes an empty list
        if not tool_calls:
            answer["text"] = call["text"]
            break

        # Put the model's own message back into the conversation: it must see its request before our result.
        assistant_message = {
            "role": "assistant",
            "content": call["message"].content,
            "tool_calls": [
                {"id": tool_call.id, "type": "function",
                 "function": {"name": tool_call.function.name, "arguments": tool_call.function.arguments}}
                for tool_call in tool_calls
            ],
        }
        reasoning_details = getattr(call["message"], "reasoning_details", None)
        if reasoning_details:   # the model's earlier thinking, which some models need in order to continue
            assistant_message["reasoning_details"] = reasoning_details
        messages.append(assistant_message)

        # Run every tool the model asked for and send each result back.
        for tool_call in tool_calls:
            try:
                arguments = json.loads(tool_call.function.arguments or "{}")   # the inputs arrive as JSON text
            except Exception:
                arguments = {}
            answer["tools_called"].append(tool_call.function.name)
            messages.append({"role": "tool", "tool_call_id": tool_call.id,
                             "content": run_tool(tool_call.function.name, arguments)})
    return answer


# ==========================================================================
# Step 15: The ruler of this notebook: score a context on the policy questions
# ==========================================================================
def is_correct(reply, question):
    """True when every pattern in the question's "check" is found in the reply (upper or lower case)."""
    reply = reply.replace("*", "")   # a model sometimes writes a number in **bold**; the stars are not part of the answer
    return all(re.search(pattern, reply, re.IGNORECASE) for pattern in question["check"])


def score(build_context, questions=None, title="", model=None, show_right=True):
    """Ask the bot every question, each with the context that build_context(question text) returns. Count the right answers.

    questions: the 20 policy questions unless you pass another list (each needs "id", "question" and "check").
    model: MODEL unless you pass SMALL_MODEL.
    """
    questions = QUESTIONS if questions is None else questions   # QUESTIONS is loaded in Exercise 1

    def answer_one(question):
        answer = ask_bot(build_context(question["question"]), model=model)
        answer.update({"id": question["id"], "question": question["question"],
                       "right": is_correct(answer["text"], question)})
        return answer

    rows = run_in_parallel(answer_one, questions)
    result = {
        "title": title,
        "model": model or MODEL,
        "rows": rows,                                              # one answer per question, in the order of the questions
        "right": sum(row["right"] for row in rows),                # True counts as 1, False as 0
        "total": len(rows),
        "tokens_in": sum(row["tokens_in"] for row in rows),        # all input tokens of this run
        "cost": sum(row["cost"] for row in rows),
        "calls": [call for row in rows for call in row["calls"]],  # every model call of this run, for the bill
    }
    # show_right=False leaves the count of right answers out of the line (for a run where it is not what is compared)
    right_part = f"{result['right']} of {result['total']} right · " if show_right else ""
    print(f"📊 {title or 'score'}: {right_part}{result['tokens_in']:,} tokens in · ${result['cost']:.4f}")
    return result


def show_wrong_answers(result, limit=5):
    """Print the answers of a run that were not right (at most `limit` of them), next to the correct answer."""
    wrong = [row for row in result["rows"] if not row["right"]]
    if not wrong:
        print("   (no wrong answers in this run)")
    for row in wrong[:limit]:
        # the correct answer, looked up among the policy questions and (once Exercise 4 ran) its follow-up questions
        correct = next((q.get("answer", "") for q in QUESTIONS + globals().get("LIMIT_QUESTIONS", []) if q["id"] == row["id"]), "")
        said = row["text"].replace("\n", " ")[:150] or f"(no answer: {row['error'] or 'the bot only called tools'})"
        print(f"   {row['id']} wrong · the bot said: {said}")
        if correct:
            print(f"   {'':<{len(row['id'])}}         correct  : {correct}")
    if len(wrong) > limit:
        print(f"   ... and {len(wrong) - limit} more")


# ==========================================================================
# Step 16: The results table that grows through the notebook
# ==========================================================================
# Exercises 4 to 9 each add one row: the score with the broken context, and the score after the fix.
RESULTS = []


def record_result(problem, broken, fixed, fix, counted, counted_broken, counted_fixed, compare_scores=True):
    """Store one row for the final table.

    broken, fixed: the two runs (results of score). fix: the name of the fix.
    counted: what shows this problem most directly, in words; counted_broken, counted_fixed: that count in the two runs.
    compare_scores=False: for an exercise that does not compare right answers; its score columns say "not compared".
    """
    row = {
        "problem": problem,
        "model": broken["model"].split("/")[-1],   # the model's name without its vendor
        "score broken": f"{broken['right']} of {broken['total']}" if compare_scores else "not compared",
        "score fixed": f"{fixed['right']} of {fixed['total']}" if compare_scores else "not compared",
        # the same scores as numbers, for the chart (None: no bar)
        "share broken": broken["right"] / max(1, broken["total"]) if compare_scores else None,
        "share fixed": fixed["right"] / max(1, fixed["total"]) if compare_scores else None,
        "tokens in, broken": broken["tokens_in"],
        "tokens in, fixed": fixed["tokens_in"],
        "what was counted": counted,
        "broken": counted_broken,
        "fixed": counted_fixed,
        "the fix": fix,
    }
    RESULTS[:] = [old for old in RESULTS if old["problem"] != problem]   # running a cell twice replaces its row
    RESULTS.append(row)
    print(f"📌 Stored for the final table: {problem} · {counted}: {counted_broken} before, {counted_fixed} after the fix")


# ==========================================================================
# Step 17: Helpers for the files you upload
# ==========================================================================
# A file counts as "here" only if it exists and really opens: a broken or half-uploaded file is asked for again.
def file_is_ready(name):
    """True if the file exists and opens as its type (JSON here); other types: not empty."""
    if not os.path.exists(name) or os.path.getsize(name) == 0:
        return False
    try:
        if name.lower().endswith(".json"):
            with open(name) as f:
                json.load(f)
        return True
    except Exception:
        return False


def describe_file(name):
    """One short line on what a file holds, so you can see the upload worked."""
    size_kb = os.path.getsize(name) / 1024
    if name.lower().endswith(".json"):
        with open(name) as f:
            data = json.load(f)
        what = f"{len(data)} entries" if isinstance(data, list) else f"{len(data)} top-level keys"
        return f"{what}, {size_kb:.1f} KB"
    return f"{size_kb:.1f} KB"


def need_file(name):
    """Make sure a file is here: ask for the upload if it is missing, stop with a clear message if it still is."""
    if not file_is_ready(name):
        print(f"📁 Please upload '{name}':")
        files.upload()
    assert file_is_ready(name), f"'{name}' is missing or does not open. Upload it again (the exact name matters)."
    print(f"📋 {name}: {describe_file(name)}")


print(f"✅ Setup done. Model: {MODEL} at reasoning effort '{EFFORT}'")
print(f"   Price: ${PRICE_IN:.2f} per million tokens in, ${PRICE_OUT:.2f} per million tokens out")
print(f"   Small model: {SMALL_MODEL} at temperature 0")
print(f"   Price: ${SMALL_PRICE_IN:.2f} per million tokens in, ${SMALL_PRICE_OUT:.2f} per million tokens out")
print(f"   Tool catalog: {len(ALL_TOOLS)} tools · about {count_tokens(json.dumps(ALL_TOOLS)):,} tokens of descriptions")

# 📁 Upload the Files (Optional, Recommended)

This notebook uses **4 files** that we give you. You find them in the same place as this notebook. Upload them once now, and every exercise will find them.

### 📄 The files

1. **`bank_fee_policy.md`**: used from Exercise 1 on
   - **Contains:** the bank's fee policy for personal accounts, in 12 numbered sections (about three pages)
2. **`fee_policy_questions.json`**: used from Exercise 1 on
   - **Contains:** 20 questions about the fee policy, each with its answer
   - **Fields:** `id`, `question`, `answer`, `section` (the policy section that holds the answer), `check` (patterns that must all be found in a correct reply), `unusual` (true for five answers that are unusual for a bank; those also have `common_value`, the value most banks use, and `common_value_check`)
3. **`bank_terms_and_conditions.md`**: used in Exercises 5 and 9
   - **Contains:** the bank's full terms and conditions, about 60 pages: the 12 sections of the fee policy spread among many other sections, with look-alike sections for other products and for an older fee schedule
4. **`bank_terms_wrong_sources.json`**: used in Exercises 5 and 9
   - **Contains:** for each of the 20 questions, the look-alike sections of the terms and conditions that hold a different number
   - **Fields:** `document` (size of the document), `sections` (every section with its `label`), `questions` (for each question id: `right_section` and a list of `wrong_sources`, each with `section`, `label`, `value` and `check`)

### 🎯 Two ways to do it

- **Option 1 (recommended):** run the cell below and upload all the files when Colab asks. The exercises then run without interruptions, and you see right away that every file opened.
- **Option 2:** skip this cell. Each exercise asks for its own file when it needs it.

**Note:** uploaded files live in this Colab session. Restarting the runtime keeps them; when the runtime disconnects, they are deleted and you upload them again (this cell tells you which ones).

In [ ]:
# ==========================================================================
# 📁 Upload the Files (optional: each exercise also asks for its own file)
# ==========================================================================

# Make sure the Setup cell ran first (it creates file_is_ready and describe_file)
assert "file_is_ready" in globals(), "Run the Setup cell at the top of the notebook first."

# Every file this notebook needs, and the exercise that uses it
REQUIRED_FILES = {
    "bank_fee_policy.md": "Exercise 1 and every exercise after it: the fee policy",
    "fee_policy_questions.json": "Exercise 1 and every exercise after it: the 20 questions",
    "bank_terms_and_conditions.md": "Exercises 5 and 9: the full terms and conditions",
    "bank_terms_wrong_sources.json": "Exercises 5 and 9: which number stands in which look-alike section",
}

# ==========================================================================
# Step 1: Check which files are already here
# ==========================================================================
# Running this cell again is safe: files that are here and open are reused, not asked for again.
ready = [name for name in REQUIRED_FILES if file_is_ready(name)]
missing = [name for name in REQUIRED_FILES if name not in ready]

print("=" * 70)
print("📁 FILES FOR THIS NOTEBOOK")
print("=" * 70)
for name in ready:
    print(f"✅ Already here: {name} ({describe_file(name)})")
for name in missing:
    print(f"📂 Missing: {name} (needed for {REQUIRED_FILES[name]})")

# ==========================================================================
# Step 2: Upload the missing files
# ==========================================================================
if missing:
    print("\n👉 Click 'Choose Files' and select the missing files")
    uploaded = files.upload()   # opens Colab's file picker; returns {file name: contents}
    for name in uploaded:
        if name in REQUIRED_FILES:
            print(f"✅ Uploaded: {name}")
        else:
            # A browser often renames a second download, e.g. "bank_fee_policy (1).md"
            print(f"⚠️ Uploaded '{name}', but this notebook expects: {', '.join(missing)}. "
                  "Rename the file on your computer and upload it again.")

    # ======================================================================
    # Step 3: Check again, by opening every file
    # ======================================================================
    still_missing = [name for name in missing if not file_is_ready(name)]
    print()
    for name in missing:
        if name not in still_missing:
            print(f"📋 {name}: {describe_file(name)}")
    if still_missing:
        print("⚠️ Still missing or not opening: " + ", ".join(still_missing))
        print("   Each exercise will ask for its file again when it needs it.")
    else:
        print("🎉 All files uploaded and checked. Every exercise can run without asking again.")
else:
    print("\n🎉 All files are already here. No upload needed.")

# 🧱 Topic 1: The Context: Everything the Model Reads in One Call

A model has no access to the bank. It does not know the bank's fees, it does not remember the customer, and it has never seen your instructions before. In every call it knows two things only: what it learned in training, and the text you send in that call. That text is the **context**. If a fact is not in the context, then for the model it does not exist.

Here is the context of one call to the bank's support bot, when a customer asks how long they have to dispute a charge. It has four parts, which this notebook calls **layers**:

| Layer | Who wrote it | What it holds in this call |
|---|---|---|
| **System prompt** | the bank's developers | *"You are the support assistant of a bank. ... Answer in one or two short sentences ..."* |
| **Policy sections** | the bank; picked for this question by code | *"## 8. Disputing a charge ... A dispute must be opened within 45 days of the date of the statement ..."* and one more section |
| **Conversation so far** | the customer and the bot, earlier | *"Does the Standard plan have a monthly fee?"* / *"No. The Standard plan has no monthly fee."* |
| **The question** | the customer, now | *"How many days does a customer have to dispute a card charge, counted from the statement date?"* |

The API takes these layers as a list of **messages**, and each message has a **role** that says who is speaking. The system prompt and the policy sections go into one message with the role `system`. Earlier turns of the conversation alternate between `user` (the customer) and `assistant` (the bot). The new question is the last `user` message.

The bot cannot carry the whole bank in every call, so code chooses what goes in. For the question above it puts in the two policy sections that share the most words with the question, and leaves the other ten out. Choosing what the model reads, in which order and how much of it, is called **context engineering**.

The context has a size limit and a price. Both are counted in **tokens**, the pieces of text a model reads (roughly three quarters of a word each).

- **The context window** is the most tokens a model can read in one call. For the larger of the two models in this notebook it is 1,000,000 tokens. You will stay far below it: its price is five times higher for a request above 100,000 tokens.
- **The token budget** is your own, smaller limit: how many tokens you decide to spend on each layer. Every token of input is paid for in every call, so a layer that is ten times larger than it needs to be costs ten times as much, on every question of every customer.

---

### 🔍 What You'll Learn

- Assemble a context from four layers as a list of messages with the roles `system`, `user` and `assistant`
- Pick the policy sections that belong to a question with code
- Count the tokens of each layer, and compare your estimate with the bill
- Measure a context on questions with known answers

### 🧠 Why This Matters

- When a bot gives a wrong answer, the first question is "what was in its context?". You can only answer it if your code builds the context in one place, in layers you can print
- The token count per layer shows where the money goes: in most products it is the documents and the history, not the question

## 🧪 Exercise 1: Assemble the Context and Count the Tokens of Each Layer

You write two small functions. `relevant_sections(question)` picks the two sections of the fee policy that match a question. `assemble(system_prompt, documents, history, question)` puts the four layers together as a list of messages. From these two you get `build_context(question)`: the function that builds the bot's context, which you will change in every later exercise.

Then you look at what you built. For one question you print the estimated tokens of each layer and their total, and you compare the total with the number on the bill of a real call. Last, you run `score(build_context)`: the bot answers all 20 policy questions with your context, and code checks each reply. That result is the **baseline** of this notebook: the score of a healthy context, which every broken and fixed context is compared with.

Here is what a question looks like in the data. The `check` is a pattern that a correct reply must contain:

```python
{"id": "Q03", "question": "How many days does a customer have to dispute a card charge, counted from the statement date?",
 "answer": "45 days from the date of the statement on which the charge first appears.", "section": 8, "check": ["...45 days..."]}
```

### 🎯 What You'll Do

1. Load the fee policy and the 20 questions, and look at both.
2. Write `relevant_sections(question)`: the two policy sections that match the question best.
3. Write `assemble(...)`: the four layers as a list of messages. Then define `build_context(question)` with it.
4. Count the tokens of each layer for one question, and compare the total with the bill of one real call.
5. Run `score(build_context)` on the 20 questions and store the result as `BASELINE`.

👉 By the end, you'll have the function that builds the bot's context, the token count of each layer, and the baseline score.

---

### ✅ You Succeed When

- The context starts with one `system` message, ends with the customer's question as a `user` message, and has the earlier turns in between
- For every one of the 20 questions, the section that holds the answer is among the two sections in the context
- The four layer counts add up to the total you print
- `BASELINE` holds 20 answers

---

### 🛠 Technical Info

- **Picking sections:** `rank_by_word_overlap(question, texts)` returns the positions of the texts, best match first. `[:2]` keeps the first two
- **A message:** `{"role": "system", "content": text}`; the roles are `"system"`, `"user"` and `"assistant"`
- **Earlier turns:** `turns_as_messages(EARLIER_TURNS)` returns them as `user` and `assistant` messages
- **Counting:** `count_tokens(text)` is an estimate (see the honest note); the exact number of a call is `call["tokens_in"]`
- **Scoring:** `score(build_context, title="...")` returns a dictionary: `right`, `total`, `tokens_in`, `cost`, and `rows` (one answer per question)
- **Honest note:** `count_tokens` cuts text the way another vendor's models do. This model cuts text a little differently, and the API adds a few tokens for the message format, so the bill shows a somewhat higher number than your estimate. The estimate is good for comparing layers. The bill is what you pay.
- **Difficulty:** 🟢 Level 1, Guided

---

### 🚀 Level Up (optional)

1. Change `SECTIONS_PER_QUESTION` to 1, then to 4, and run the cell again. What happens to the tokens per call and to the baseline? One section is cheaper, and a wrong pick then leaves the bot without the answer.
2. Put the full policy (all 12 sections) into every context and compare the tokens and the score with the two-section version.

In [ ]:
# ==========================================================================
# 🧱 Exercise 1: Assemble the Context and Count the Tokens of Each Layer
# ==========================================================================

# Make sure the Setup cell ran first (it creates score, count_tokens and the other helpers)
assert "score" in globals(), "Run the Setup cell at the top of the notebook first."

# ==========================================================================
# Step 1: Load the fee policy and the 20 questions (upload the files if they are not here yet)
# ==========================================================================
need_file("bank_fee_policy.md")
need_file("fee_policy_questions.json")

with open("bank_fee_policy.md") as f:
    POLICY_TEXT = f.read()
with open("fee_policy_questions.json") as f:
    QUESTIONS = json.load(f)   # a list of 20 dictionaries: id, question, answer, section, check, unusual

# Cut the policy into its 12 sections. Every section starts with a heading line like "## 3. Cash withdrawals at ATMs".
POLICY_SECTIONS = []
for part in re.split(r"(?m)^## ", POLICY_TEXT)[1:]:   # (?m)^ means "at the start of a line"; [1:] skips the title before section 1
    heading = part.split("\n", 1)[0]                   # the first line, e.g. "3. Cash withdrawals at ATMs"
    number, title = heading.split(". ", 1)
    POLICY_SECTIONS.append({"number": int(number), "title": title, "text": "## " + part.strip()})

# ==========================================================================
# Step 2: Look at the data before using it
# ==========================================================================
print()
print("=" * 70)
print("📊 THE FEE POLICY: 12 SECTIONS")
print("=" * 70)
for section in POLICY_SECTIONS:
    print(f"   {section['number']:>2}. {section['title']:<45}{count_tokens(section['text']):>5} tokens")
print(f"   The whole policy: about {count_tokens(POLICY_TEXT):,} tokens")
print()
print("=" * 70)
print("📊 THE QUESTIONS (2 of 20)")
print("=" * 70)
for question in QUESTIONS[:2]:
    print(f"{question['id']}: {question['question']}")
    print(f"     correct answer: {question['answer']}   (policy section {question['section']})")

# ==========================================================================
# Step 3: Pick the policy sections that belong to a question
# ==========================================================================
# The bot does not get the whole policy, only the sections that match the question best.
SECTIONS_PER_QUESTION = 2


def relevant_sections(question, sections=None, how_many=SECTIONS_PER_QUESTION):
    """Return the texts of the sections that share the most meaningful words with the question."""
    sections = POLICY_SECTIONS if sections is None else sections
    order = rank_by_word_overlap(question, [section["text"] for section in sections])   # positions, best match first
    return [sections[position]["text"] for position in order[:how_many]]


# ==========================================================================
# Step 4: Assemble the four layers into a list of messages
# ==========================================================================
# Layer 1 and 2 share the system message: the standing instructions, then the documents inside <documents> tags.
# The tags only mark where the documents start and end, so the model can tell them from the instructions.
def assemble(system_prompt, documents, history, question):
    """Put the four layers together: system prompt, documents (a list of texts), history (messages), question."""
    system_text = system_prompt
    if documents:
        system_text += "\n\n<documents>\n" + "\n\n".join(documents) + "\n</documents>"
    messages = [{"role": "system", "content": system_text}]   # layers 1 and 2: instructions and documents
    messages += history                                        # layer 3: the conversation so far
    messages.append({"role": "user", "content": question})     # layer 4: the customer's question, always last
    return messages


def build_context(question):
    """The bot's context for one question: system prompt, two policy sections, the earlier turns, the question."""
    return assemble(SYSTEM_PROMPT, relevant_sections(question), turns_as_messages(EARLIER_TURNS), question)


# ==========================================================================
# Step 5: Count the tokens of each layer, for one question
# ==========================================================================
# We take the question about disputes (Q03) and count each layer by itself.
example = QUESTIONS[2]
example_context = build_context(example["question"])
layer_tokens = {
    "system prompt": count_tokens(SYSTEM_PROMPT),
    "policy sections": count_tokens("\n\n".join(relevant_sections(example["question"]))),
    "conversation so far": sum(count_tokens(message["content"]) for message in turns_as_messages(EARLIER_TURNS)),
    "the question": count_tokens(example["question"]),
}
total_tokens = sum(layer_tokens.values())

print()
print("=" * 70)
print("📊 THE CONTEXT OF ONE CALL, LAYER BY LAYER")
print("=" * 70)
print(f"Question: {example['question']}")
for message in example_context:
    first_line = message["content"].replace("\n", " ")[:90]
    print(f"   {message['role']:<10} {first_line}...")
print()
for layer, tokens in layer_tokens.items():
    bar = "█" * round(40 * tokens / total_tokens)   # a bar that shows the layer's share of the context
    print(f"   {layer:<22}{tokens:>5} tokens  {tokens / total_tokens:>4.0%}  {bar}")
print(f"   {'total (estimate)':<22}{total_tokens:>5} tokens")

# ==========================================================================
# Step 6: Compare the estimate with the bill of one real call
# ==========================================================================
# The bill has the exact number of input tokens. It is a little higher than the estimate (see the honest note).
example_call = call_model(example_context)
print()
print(f"The bot's answer : {textwrap.fill(example_call['text'], width=100)}")
print(f"Your estimate    : {total_tokens} tokens in")
print(f"On the bill      : {example_call['tokens_in']} tokens in")
show_bill([example_call], "one call")

# ==========================================================================
# Step 7: The baseline: score the context on the 20 questions
# ==========================================================================
# score() calls build_context once per question, asks the bot, and checks each reply with the question's pattern.
print()
print("=" * 70)
print("📊 THE BASELINE")
print("=" * 70)
BASELINE = score(build_context, title="baseline (a healthy context)")
show_wrong_answers(BASELINE)
print(f"Tokens in per question: about {BASELINE['tokens_in'] // BASELINE['total']:,}")
show_bill(BASELINE["calls"], "Exercise 1, the 20 questions")

# ==========================================================================
# Step 8: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
history_messages = turns_as_messages(EARLIER_TURNS)
shape_ok = (example_context[0]["role"] == "system" and example_context[-1] == {"role": "user", "content": example["question"]}
            and example_context[1:-1] == history_messages
            and [message["role"] for message in example_context].count("system") == 1)
print("✅ The context starts with one system message, then the earlier turns, then the question" if shape_ok
      else "❌ The context should be: one system message, the earlier turns, the question as the last user message")
missing_section = [question["id"] for question in QUESTIONS
                   if not any(text.startswith(f"## {question['section']}. ")
                              for text in relevant_sections(question["question"]))]
print("✅ For all 20 questions, the section with the answer is among the two sections in the context"
      if not missing_section and len(relevant_sections(example["question"])) == 2
      else f"❌ The section with the answer is missing for: {', '.join(missing_section) or 'none'} "
           f"(and each context should hold {SECTIONS_PER_QUESTION} sections)")
print(f"✅ The four layer counts add up to the total ({total_tokens} tokens)" if total_tokens == sum(layer_tokens.values())
      and len(layer_tokens) == 4 else "❌ total_tokens should be the sum of the four layer counts")
print("✅ BASELINE holds 20 answers" if BASELINE["total"] == 20 and len(BASELINE["rows"]) == 20
      else f"❌ BASELINE should hold 20 answers, it holds {BASELINE['total']}")
if any(row["error"] for row in BASELINE["rows"]):
    print("🤔 Some calls failed and have no answer (see the bill above). Run the cell again.")
elif BASELINE["right"] >= 18:
    print(f"🤔 The baseline is {BASELINE['right']} of 20. A healthy context gets nearly everything right, as it should: "
          "the answer to every question is in one of its two sections.")
else:
    print(f"🤔 The baseline is {BASELINE['right']} of 20, lower than expected for a healthy context. Read the wrong "
          "answers above: a reply that states the right number in other words can miss the pattern. Run the cell again "
          "to see whether the same questions fail.")
print("🧠 The context is the only thing the model knows about your bank. You decide what is in it, layer by layer.")

# 🔁 Topic 2: A Model Remembers Nothing Between Calls, So Every Turn Resends the Whole Conversation

In a chat it looks as if the bot remembers what you said a minute ago. It does not. A model is **stateless**: it keeps no memory between calls. Each call starts from nothing, and the model knows only the context of that one call.

So how does a bot answer "And what about the Plus plan?" when the plan question was asked two turns ago? Your code keeps the conversation in a list, the **conversation history**, and sends **all of it again** with every new question. Here is what goes out in the first three turns of a chat with the bank's bot:

| Turn | What your code sends | Of which is new |
|---|---|---|
| 1 | system prompt and policy sections · question 1 | question 1 |
| 2 | system prompt and policy sections · question 1 · answer 1 · question 2 | question 2 |
| 3 | system prompt and policy sections · question 1 · answer 1 · question 2 · answer 2 · question 3 | question 3 |

Each turn carries everything from the turns before it. The model reads question 1 three times in three turns, and you pay for it three times.

That is why the cost of a conversation grows faster than its length. If every turn adds about 100 tokens of question and answer, turn 1 sends about 100 tokens of conversation, turn 2 about 200, turn 10 about 1,000. Ten turns together send 100 + 200 + ... + 1,000 = 5,500 tokens of conversation, for a chat that is only 1,000 tokens long. A conversation twice as long costs about four times as much.

In code, the memory of a chat bot is two lines after every call:

```python
history.append({"role": "user", "content": question})       # what the customer asked
history.append({"role": "assistant", "content": reply})     # what the bot answered
```

Everything a chat product seems to "remember" works this way, or is a variation of it: the history is stored by code and put back into the context.

---

### 🔍 What You'll Learn

- Keep a conversation history in code and resend it with every turn
- Read a complete request and find the earlier turns in it
- Measure how the input tokens and the cost grow from turn to turn
- Compare a conversation with the same questions asked in separate calls

### 🧠 Why This Matters

- A support chat that runs for forty turns can cost far more than forty single questions, and the bill arrives without any change in your code
- Because the history is put back by your code, your code can also shorten it, summarise it or correct it. Exercise 4 uses exactly that

## 🧪 Exercise 2: Hold a Ten-Turn Conversation and Plot the Cost of Each Turn

A customer asks the bank's bot ten questions in one chat. You write the loop that holds the conversation: for each question it builds the context with the history so far, calls the model, and adds the question and the reply to the history.

At turn five you print the complete request, message by message, and see that it contains turns one to four. After the last turn you plot the input tokens of each turn, split into the part that stays the same size (system prompt and policy sections) and the part that grows (the history), with the cost of each turn next to it.

Then you ask the same ten questions again, each in a separate call with an empty history, and compare the two totals. The difference is what the resending cost.

### 🎯 What You'll Do

1. Write `build_context(question, history)`: the context from Exercise 1, with the history as an input.
2. Hold the ten-turn conversation, adding each question and reply to the history.
3. Print the complete request of turn five.
4. Plot the input tokens and the cost of each turn.
5. Ask the same ten questions in ten separate calls, and compare tokens and cost.

👉 By the end, you'll have a chart of a conversation's cost per turn, and the price of resending in dollars.

---

### ✅ You Succeed When

- The conversation has 10 turns, and the history holds 20 messages at the end
- The request of turn five holds 10 messages: the system message, turns one to four, and question five
- The history part of the context is larger at every turn than at the turn before
- The conversation used more input tokens than the ten separate calls

---

### 🛠 Technical Info

- **The history:** a list of messages; after each call, append the question (`"user"`) and the reply (`"assistant"`)
- **One call:** `call_model(messages)` returns `call["text"]`, `call["tokens_in"]` and `call["cost"]`
- **Honest note:** the turns of a conversation cannot run at the same time, because each one needs the reply of the one before. So this cell takes longer than the others: ten calls, one after another.
- **Honest note about the chart:** the policy sections change with each question, and some sections are longer than others. So the bars do not rise in perfectly even steps. The history part rises at every turn.
- **Difficulty:** 🟢 Level 1, Guided

---

### 🚀 Level Up (optional)

1. Keep only the last three turns in the context (`history[-6:]`) and run the conversation again. How much does it save, and which kind of question would the bot now get wrong?
2. Work out the cost of a 40-turn conversation from your numbers: take the average growth per turn and add it up.

In [ ]:
# ==========================================================================
# 🔁 Exercise 2: Hold a Ten-Turn Conversation and Plot the Cost of Each Turn
# ==========================================================================

# Make sure the Setup cell and Exercise 1 ran first (Exercise 1 creates assemble, relevant_sections and QUESTIONS)
assert "assemble" in globals(), "Run the Setup cell and Exercise 1 first."

# ==========================================================================
# Step 1: The context builder, now with the history as an input
# ==========================================================================
# The same four layers as in Exercise 1. When no history is given, the two earlier turns from Setup are used,
# so score(build_context) still works as before.
def build_context(question, history=None):
    """The bot's context for one question: system prompt, two policy sections, the history, the question."""
    if history is None:
        history = turns_as_messages(EARLIER_TURNS)
    return assemble(SYSTEM_PROMPT, relevant_sections(question), history, question)


# ==========================================================================
# Step 2: Hold the conversation
# ==========================================================================
# Ten questions of one customer, asked one after another. The history starts empty and grows by two messages per turn.
conversation_questions = [question["question"] for question in QUESTIONS[:10]]
history = []
turns = []                 # what we measure at each turn
request_at_turn_five = None

for turn_number, question in enumerate(conversation_questions, start=1):   # enumerate counts the turns: 1, 2, 3, ...
    context = build_context(question, history)
    if turn_number == 5:
        request_at_turn_five = list(context)   # keep a copy of the complete request, to print it below
    call = call_model(context)
    turns.append({
        "turn": turn_number,
        "tokens_in": call["tokens_in"],
        "cost": call["cost"],
        "history_tokens": sum(count_tokens(message["content"]) for message in history),   # estimate of the history part
        "call": call,
    })
    # The bot's memory: add the question and the reply to the history, so the next turn carries them.
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": call["text"]})

print("=" * 70)
print("📊 THE CONVERSATION, TURN BY TURN")
print("=" * 70)
print(f"{'turn':<6}{'messages sent':>14}{'tokens in':>11}{'of which history':>18}{'cost':>12}")
for turn in turns:
    messages_sent = 1 + 2 * (turn["turn"] - 1) + 1   # the system message, two messages per earlier turn, the question
    print(f"{turn['turn']:<6}{messages_sent:>14}{turn['tokens_in']:>11,}{turn['history_tokens']:>18,}{turn['cost']:>12.6f}")

# ==========================================================================
# Step 3: Print the complete request of turn five
# ==========================================================================
# Every message of the request, with its role. Turns one to four are all in it.
print()
print("=" * 70)
print("📊 THE COMPLETE REQUEST AT TURN FIVE")
print("=" * 70)
for position, message in enumerate(request_at_turn_five):
    text = message["content"].replace("\n", " ")
    print(f"   [{position}] {message['role']:<10} {text[:95]}{'...' if len(text) > 95 else ''}")

# ==========================================================================
# Step 4: Plot the input tokens and the cost of each turn
# ==========================================================================
# Each bar is the input of one turn: grey for the part that is sent fresh (system prompt, sections, question),
# blue for the history that is sent again. The line is the cost of the turn.
turn_numbers = [turn["turn"] for turn in turns]
history_part = [min(turn["history_tokens"], turn["tokens_in"]) for turn in turns]
fresh_part = [turn["tokens_in"] - part for turn, part in zip(turns, history_part)]   # zip walks through both lists side by side

figure, tokens_axis = plt.subplots(figsize=(8, 4))
tokens_axis.bar(turn_numbers, fresh_part, color="lightgrey", label="system prompt, policy sections, question")
tokens_axis.bar(turn_numbers, history_part, bottom=fresh_part, color="tab:blue", label="history, sent again")
tokens_axis.set_xlabel("Turn of the conversation")
tokens_axis.set_ylabel("Input tokens of the turn")
tokens_axis.set_xticks(turn_numbers)
tokens_axis.legend(loc="upper left")
cost_axis = tokens_axis.twinx()   # a second y-axis on the right, for the cost
cost_axis.plot(turn_numbers, [turn["cost"] * 100 for turn in turns], color="tab:red", marker="o")
cost_axis.set_ylabel("Cost of the turn (US cents)", color="tab:red")
tokens_axis.set_title("Every turn resends the history, so every turn has more input than the one before")
plt.show()

# ==========================================================================
# Step 5: The same ten questions in ten separate calls
# ==========================================================================
# Each question alone, with an empty history: no question carries the ones before it.
separate_calls = run_in_parallel(lambda question: call_model(build_context(question, history=[])), conversation_questions)

conversation_tokens = sum(turn["tokens_in"] for turn in turns)
conversation_cost = sum(turn["cost"] for turn in turns)
separate_tokens = sum(call["tokens_in"] for call in separate_calls)
separate_cost = sum(call["cost"] for call in separate_calls)

print()
print("=" * 70)
print("📊 ONE CONVERSATION AGAINST TEN SEPARATE CALLS")
print("=" * 70)
print(f"{'':<28}{'tokens in':>12}{'cost':>12}")
print(f"{'ten-turn conversation':<28}{conversation_tokens:>12,}{conversation_cost:>12.5f}")
print(f"{'ten separate calls':<28}{separate_tokens:>12,}{separate_cost:>12.5f}")
if separate_tokens:
    print(f"The conversation read {conversation_tokens / separate_tokens:.1f} times as many input tokens for the same ten questions.")
show_bill([turn["call"] for turn in turns] + separate_calls, "Exercise 2")

# ==========================================================================
# Step 6: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ The conversation has 10 turns, and the history holds 20 messages" if len(turns) == 10 and len(history) == 20
      and [message["role"] for message in history] == ["user", "assistant"] * 10
      else f"❌ Expected 10 turns and 20 history messages (user, assistant, ...), got {len(turns)} turns and {len(history)} messages")
earlier_questions_in_request = [message["content"] for message in (request_at_turn_five or []) if message["role"] == "user"]
print("✅ The request of turn five holds 10 messages: the system message, turns one to four, and question five"
      if request_at_turn_five and len(request_at_turn_five) == 10 and earlier_questions_in_request == conversation_questions[:5]
      else "❌ The request of turn five should hold the system message, turns one to four and question five (10 messages)")
history_grows = all(later["history_tokens"] > earlier["history_tokens"] for earlier, later in zip(turns, turns[1:]))
any_failed = any(turn["call"]["error"] for turn in turns) or any(call["error"] for call in separate_calls)
if any_failed:
    print("🤔 Some calls failed and have no reply (see the bill above), so the totals are incomplete. Run the cell again.")
else:
    print("✅ The history part of the context is larger at every turn than at the turn before" if history_grows
          else "❌ The history should grow at every turn: add the question and the reply to it after each call")
    print("✅ The conversation used more input tokens than the ten separate calls" if conversation_tokens > separate_tokens
          else "❌ The conversation should use more input tokens than ten separate calls: is the history sent with every turn?")
print("🧠 The model remembers nothing. Your code resends the conversation, and you pay for every resent token.")

# 🗄️ Topic 3: Prompt Caching: Paying Less for a Beginning That Stays the Same

In Exercise 2 you saw that most of what a call sends was already sent in the call before. Providers offer a discount for exactly that, called **prompt caching**: when the **beginning** of a request is identical to the beginning of an earlier request, the provider reuses the work it already did on that part and charges much less for it. For the larger model in this notebook, tokens read from the cache cost a tenth of the normal input price.

The important word is *beginning*. The cache compares requests from the first token on and stops at the first difference. Everything before the difference can be reused. Everything after it is new, even if it is word for word the same as last time. Take a bot that tells the model the current time, and look at two ways to order its system message:

| | The request starts with | The first difference between two calls | What can be reused |
|---|---|---|---|
| **Time first** | `Current time: 18:06:42` · instructions · the fee policy | the first line | nothing |
| **Time last** | instructions · the fee policy · `Current time: 18:06:42` | after the fee policy | the instructions and the fee policy |

The two requests hold the same text. Only the order differs, and with it the bill. The rule that follows is called a **stable prefix**: put what never changes at the top (instructions, documents, examples), and what changes with every call at the end (the time, the customer's name, the new question).

Three details decide whether the cache is used with this model:

- **You mark where the stable part ends.** The system message is sent as a list of text blocks, and the block that ends the stable part carries `"cache_control": {"type": "ephemeral"}`. The provider stores everything up to and including that block. (Some other vendors' models cache automatically, without a marker. The rule about the stable prefix is the same.)
- **The stable part must be long enough.** For this model the minimum is 512 tokens. That is why the bot in this exercise carries the whole fee policy, about 2,000 tokens.
- **The cache forgets.** An entry lives for five minutes and is refreshed each time it is read. The first call **writes** the entry and pays a little more for those tokens (1.25 times the input price). Later calls **read** it at a tenth of the price.

Each response tells you what happened, in its `usage` field: `cached_tokens` is how many input tokens were read from the cache, and `cache_write_tokens` is how many were written into it.

---

### 🔍 What You'll Learn

- Order a context so that its beginning is identical from call to call
- Mark the end of the stable part with `cache_control`
- Read the cached tokens of a response and work out what the cache saved

### 🧠 Why This Matters

- One value that changes at the top of a prompt (a timestamp, a request id, the customer's name) silently turns the cache off for everything below it, and the only sign is the bill
- A bot with a long, stable system prompt and many calls per day pays up to ten times less for that part when the order is right

## 🧪 Exercise 3: Put the Current Time First, Then Last, and Read the Cached Tokens

The bank wants the bot to know the current time. You build the context in two orders and make five calls with each, one after another, with five different questions.

In the first order the time stands at the very top of the system message, before the instructions and the fee policy. In the second order the instructions and the policy come first, and the time is the last block. In both orders the stable part is marked for the cache in the same way, so the only difference between the two sets of calls is where the time stands.

For every call you read `cached_tokens` and `cache_write_tokens` and the cost. What you should see: with the time first, no call reads anything from the cache, and every call pays to write an entry that is never used. With the time last, the first call writes the entry and the next four read it.

### 🎯 What You'll Do

1. Build the stable part: the system prompt and the whole fee policy.
2. Write `context_time_first(question)`: one block that starts with the time.
3. Write `context_time_last(question)`: the stable block with the cache marker, then a second block with the time.
4. Make five calls with each order, and print the cached tokens and the cost of every call.
5. Compare the two sets: cached tokens, cost, and what the difference means for 10,000 calls a day.

👉 By the end, you'll have seen the cache stay empty with the time on top, and fill and be read with the time at the end.

---

### ✅ You Succeed When

- In the time-first context, the system message is one block that starts with the time and carries the cache marker
- In the time-last context, the first block is exactly the stable part and carries the cache marker, and the time is in the block after it
- You made 5 calls with each order
- 🤔 (depends on the provider in your run) the time-first calls read 0 tokens from the cache, and the time-last calls after the first one read the stable part from it

---

### 🛠 Technical Info

- **A system message in blocks:** `{"role": "system", "content": [{"type": "text", "text": ...}, {"type": "text", "text": ...}]}`
- **The cache marker:** add `"cache_control": {"type": "ephemeral"}` to the block that ends the stable part. "Ephemeral" means short-lived: the entry is kept for five minutes
- **Reading the result:** `call["cached_tokens"]` and `call["cache_write_tokens"]` (the setup's `call_model` copies them from `usage["prompt_tokens_details"]`)
- **Honest note:** the calls run one after another on purpose. A cache entry exists only after the first call has been answered, so five calls sent at the same moment would all miss it.
- **Recommended setup:** this exercise runs on `anthropic/claude-haiku-5.5`, because its cache is switched on by a marker in the request, so you can see exactly where the stable part ends.
- **Honest note about your numbers:** if you run this cell twice within five minutes, the first time-last call also reads from the cache, because the entry of your earlier run is still there. The time-first calls stay at zero however often you run them.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Put the customer's question into the same block as the stable part, before the policy, and make five calls with five questions. What do the cached tokens show, and why?
2. Wait six minutes and run the time-last calls again. The first call writes the entry again: the old one has expired.

In [ ]:
# ==========================================================================
# 🗄️ Exercise 3: Put the Current Time First, Then Last, and Read the Cached Tokens
# ==========================================================================

# Make sure the Setup cell and Exercise 1 ran first (Exercise 1 loads the fee policy and the questions)
assert "POLICY_TEXT" in globals(), "Run the Setup cell and Exercise 1 first."

# ==========================================================================
# Step 1: The stable part: what is the same in every call
# ==========================================================================
# The system prompt and the whole fee policy. The cache needs at least 512 tokens of stable text for this model.
STABLE_PART = SYSTEM_PROMPT + "\n\n<documents>\n" + POLICY_TEXT.strip() + "\n</documents>"
CACHE_MARKER = {"type": "ephemeral"}   # "store everything up to here"; ephemeral = kept for five minutes
print(f"The stable part: about {count_tokens(STABLE_PART):,} tokens")


def time_line():
    """The current time as one line of text. It is different in every call (it includes milliseconds)."""
    return "Current time: " + datetime.now().isoformat(sep=" ", timespec="milliseconds")


# ==========================================================================
# Step 2: Order one: the time at the very top
# ==========================================================================
# One block: the time, then the stable part. The block changes with every call, from its first line on.
def context_time_first(question):
    """The context with the current time before everything else."""
    system_blocks = [
        {"type": "text", "text": time_line() + "\n\n" + STABLE_PART, "cache_control": CACHE_MARKER},
    ]
    return [{"role": "system", "content": system_blocks}, {"role": "user", "content": question}]


# ==========================================================================
# Step 3: Order two: the stable part first, the time after it
# ==========================================================================
# Two blocks: the stable part, marked for the cache, and then the time. The first block never changes.
def context_time_last(question):
    """The context with the stable part first and the current time after it."""
    system_blocks = [
        {"type": "text", "text": STABLE_PART, "cache_control": CACHE_MARKER},
        {"type": "text", "text": time_line()},
    ]
    return [{"role": "system", "content": system_blocks}, {"role": "user", "content": question}]


# ==========================================================================
# Step 4: Five calls with each order, one after another
# ==========================================================================
# Five different questions, the same five for both orders.
cache_questions = [question["question"] for question in QUESTIONS[10:15]]


def five_calls(make_context, title):
    """Make one call per question, one after another, and print what the cache did in each."""
    calls = []
    print(f"\n{title}")
    print(f"   {'call':<6}{'tokens in':>10}{'read from cache':>17}{'written to cache':>18}{'cost':>12}")
    for number, question in enumerate(cache_questions, start=1):
        call = call_model(make_context(question))
        calls.append(call)
        print(f"   {number:<6}{call['tokens_in']:>10,}{call['cached_tokens']:>17,}{call['cache_write_tokens']:>18,}{call['cost']:>12.6f}")
    return calls


print()
print("=" * 70)
print("📊 WHAT THE CACHE DID IN EACH CALL")
print("=" * 70)
time_first_calls = five_calls(context_time_first, "Time first: the beginning changes with every call")
time_last_calls = five_calls(context_time_last, "Time last: the beginning stays the same")

# ==========================================================================
# Step 5: Compare the two sets of calls
# ==========================================================================
cached_first = sum(call["cached_tokens"] for call in time_first_calls)
cached_last = sum(call["cached_tokens"] for call in time_last_calls)
cost_first = sum(call["cost"] for call in time_first_calls)
cost_last = sum(call["cost"] for call in time_last_calls)

print()
print("=" * 70)
print("📊 TIME FIRST AGAINST TIME LAST")
print("=" * 70)
print(f"{'':<14}{'tokens read from cache':>24}{'cost of 5 calls':>18}")
print(f"{'time first':<14}{cached_first:>24,}{cost_first:>18.6f}")
print(f"{'time last':<14}{cached_last:>24,}{cost_last:>18.6f}")
# What the input of ONE call costs once the cache is warm: the stable part at the cache price, against the full price
stable_tokens = count_tokens(STABLE_PART)
per_call_full = stable_tokens * PRICE_IN / 1_000_000
per_call_cached = stable_tokens * PRICE_CACHE_READ / 1_000_000
print(f"\nThe stable part of one call (about {stable_tokens:,} tokens): ${per_call_full:.6f} at the full price, "
      f"${per_call_cached:.6f} from the cache.")
print(f"At 10,000 calls a day that is ${per_call_full * 10_000:.2f} against ${per_call_cached * 10_000:.2f} per day, "
      "for the same text in a different order.")
show_bill(time_first_calls + time_last_calls, "Exercise 3")

# ==========================================================================
# Step 6: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
first_blocks = context_time_first("test")[0]["content"]
last_blocks = context_time_last("test")[0]["content"]
print("✅ Time first: the system message is one block that starts with the time and carries the cache marker"
      if len(first_blocks) == 1 and first_blocks[0]["text"].startswith("Current time:")
      and first_blocks[0]["text"].endswith(STABLE_PART) and first_blocks[0].get("cache_control") == CACHE_MARKER
      else "❌ Time first: expected one block: the time, then the stable part, with cache_control on the block")
print("✅ Time last: the first block is exactly the stable part with the cache marker, and the time is in the next block"
      if len(last_blocks) == 2 and last_blocks[0]["text"] == STABLE_PART and last_blocks[0].get("cache_control") == CACHE_MARKER
      and last_blocks[1]["text"].startswith("Current time:") and "cache_control" not in last_blocks[1]
      else "❌ Time last: expected two blocks: the stable part with cache_control, then the time without it")
print("✅ You made 5 calls with each order" if len(time_first_calls) == 5 and len(time_last_calls) == 5
      else f"❌ Expected 5 and 5 calls, got {len(time_first_calls)} and {len(time_last_calls)}")
if any(call["error"] for call in time_first_calls + time_last_calls):
    print("🤔 Some calls failed (see the bill above). Run the cell again.")
elif cached_first == 0 and all(call["cached_tokens"] > 0 for call in time_last_calls[1:]):
    print(f"🤔 As expected: with the time first, no call read from the cache. With the time last, calls 2 to 5 read "
          f"{time_last_calls[-1]['cached_tokens']:,} tokens each from it.")
elif cached_first == 0 and cached_last == 0:
    print("🤔 No call read from the cache in your run, in either order. Your code can still be right: the cache is a "
          "service of the provider that answered your calls, and not every provider behind OpenRouter offers it for "
          "this model. Run the cell again. If it stays at zero, the order you built is still the one to use: it is "
          "what makes the cache possible.")
else:
    print(f"🤔 Tokens read from the cache: {cached_first:,} with the time first, {cached_last:,} with the time last. "
          "The expected picture is zero with the time first and the stable part in calls 2 to 5 with the time last. "
          "Run the cell again and compare.")
print("🧠 The cache reuses the beginning of a request up to the first difference. Stable text first, changing text last.")

# ☠️ Topic 4: Poisoning: A Wrong Fact Gets Into the Context and Stays

The first three exercises were about what a context costs. The next five are about what makes a context give wrong answers, even though the right facts are in it. The first way is called **context poisoning**: a false statement gets into the context, and from then on the model can treat it as true.

Here is how it happens to the bank's bot. The policy says the daily ATM limit on the Standard plan is \$500. Somewhere in a long chat, the bot gets one answer wrong:

| Turn | Who | What was said |
|---|---|---|
| 3 | customer | *"What is my daily limit for ATM withdrawals? I am on the Standard plan."* |
| 3 | bot | *"Your daily limit is \$1,000."* (wrong: the policy says \$500) |
| 7 | customer | *"I need \$800 in cash today. Can I take it all out at an ATM?"* |
| 7 | bot | *"Yes. Your daily limit is \$1,000, so you can withdraw \$800 today."* |

The mistake at turn 3 was one bad answer. The damage comes afterwards. You saw in Exercise 2 that every turn resends the whole history, so at turn 7 the model reads the sentence "Your daily limit is \$1,000" again, and it reads it as something **the bot itself said**. A model has no way to tell which of its earlier replies were right. Its own words in the history look like established facts, and it tends to stay consistent with them. The customer goes to the ATM and the withdrawal is declined.

A wrong fact can get into the history in several ways:

- **The model made a mistake** in an earlier reply, as above.
- **The customer said it** ("I was told my limit is \$1,000"), and the bot agreed instead of checking.
- **A tool or a summary brought it in**: a tool returned outdated data, or an automatic summary of a long conversation changed a number.

The fix follows from where the damage happens. The history is written by your code, so your code can check a reply **before it is saved**. Every amount the bot states must be found in the bank's policy. A reply with a number that the policy does not contain is not saved, so it cannot come back in later turns.

---

### 🔍 What You'll Learn

- Recognise poisoning: one wrong statement in the history that later answers repeat
- Check a reply against a source with code before it enters the history
- Measure a failure and its fix on the same questions

### 🧠 Why This Matters

- In a long conversation or an agent that works for an hour, one early mistake can decide everything after it, because every later step reads it again
- A check at the moment of saving is cheap and runs without a model call. Cleaning up after the fact means finding out which of a thousand later answers were affected

## 🧪 Exercise 4: Poison the History With a Wrong Limit, Then Check Replies Before They Are Saved

You insert one earlier turn into the bot's conversation history, in which the bot said "Your daily limit is \$1,000". The policy sections in the context still say \$500 wherever the question is about ATMs. So the context now holds the right number and the wrong one.

**The comparison.** You measure the bot twice on the same 25 questions: the 20 policy questions, plus five follow-up questions about the limit, written the way customers write in a chat: short, and relying on what was said before ("So can I take out \$800 in cash today?"). First with the poisoned history. Then with the fix: a function that checks the amounts in every bot reply against the policy text before the reply is saved, and leaves the turn out if an amount is not in the policy. For each run you count the right answers, the right answers among the five follow-up questions, and the answers that repeat \$1,000.

Why five follow-up questions? A wrong fact does harm where it is relevant, and among the 20 policy questions only one asks about the daily limit. A follow-up question is also where the history matters most: "that limit" only means something because of the earlier turns, so the model leans on them.

This exercise runs on the small model (`SMALL_MODEL`). The larger model corrected the wrong limit in every answer in our runs ("the limit is \$500, not \$1,000 as I said earlier"), so with it there would be nothing to measure.

### 🎯 What You'll Do

1. Add the poisoned turn to the earlier turns, and look at the five follow-up questions.
2. Run the 25 questions with the poisoned history and count the answers that repeat \$1,000.
3. Write the check: find the dollar amounts in a reply and test whether the policy contains each of them.
4. Write `checked_turns(turns)`: keep only the turns whose bot reply passes the check.
5. Run the same 25 questions with the checked history, and compare the two runs.

👉 By the end, you'll have a history that a wrong number cannot enter, and two counts that show what that is worth.

---

### ✅ You Succeed When

- The poisoned history contains the bot reply with \$1,000
- Your check rejects "Your daily limit is \$1,000." and accepts a reply with \$500
- The checked history holds the two clean turns and no \$1,000
- Both runs have 25 answers, and the result is stored in `RESULTS`

---

### 🛠 Technical Info

- **Finding amounts:** `re.findall(pattern, text)` returns every match of a pattern as a list. The pattern for a dollar amount is given
- **"All of them":** `all(condition for item in items)` is `True` when the condition holds for every item
- **A different history:** `build_context(question, history=...)` from Exercise 2 takes the history as messages; `turns_as_messages(turns)` makes them from (customer, bot) pairs
- **Other questions, the small model:** `score(build_context, questions=..., title=..., model=SMALL_MODEL)`
- **Honest note about the check:** it is deliberately simple. It catches a number that appears nowhere in the policy. It does not catch a number that is in the policy but belongs to another fee (the bot saying "\$500" where "\$2,000" is right). It also rejects a harmless reply that repeats an amount the customer named. A stricter check compares with the one section the answer came from (Level Up 1).
- **Honest note about the result:** in our runs the small model got 3 or 4 of the 5 follow-up questions right with the poisoned history (it allowed the \$800 withdrawal, or it stated the \$1,000 limit) and 5 of 5 with the checked history. The 20 policy questions are hardly affected, because the wrong fact is not relevant to them (this model misreads one of them in some runs, whatever the history). So read the column for the follow-up questions. If the bot follows the wrong limit in no answer in your run, your code is still right: the check then guarantees what the model did by itself this time.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Make the check stricter: accept an amount only if it appears in one of the two sections that match the customer's message of that turn.
2. Instead of leaving a rejected turn out, replace the bot's reply with "I am not sure about that amount. Let me check the policy." and run again. Which version would you want in a real chat?

In [ ]:
# ==========================================================================
# ☠️ Exercise 4: Poison the History With a Wrong Limit, Then Check Replies Before They Are Saved
# ==========================================================================

# Make sure the Setup cell and Exercises 1 and 2 ran first (Exercise 2 creates build_context with a history input)
assert "conversation_questions" in globals(), "Run the Setup cell and Exercises 1 and 2 first."

# ==========================================================================
# Step 1: The poisoned turn
# ==========================================================================
# One earlier turn in which the bot stated a wrong limit. The policy says $500 for the Standard plan.
POISONED_TURN = ("What is my daily limit for ATM withdrawals? I am on the Standard plan.",
                 "Your daily limit is $1,000.")
RAW_TURNS = EARLIER_TURNS + [POISONED_TURN]   # the two clean turns from Setup, then the poisoned one

print("=" * 70)
print("📊 THE HISTORY WITH THE POISONED TURN")
print("=" * 70)
for message in turns_as_messages(RAW_TURNS):
    print(f"   {message['role']:<10} {message['content']}")

# ==========================================================================
# Step 2: Five follow-up questions in which the daily limit matters
# ==========================================================================
# They are written the way customers write in a chat: short, and relying on the turns before.
# The right limit is $500 in all five. The check is the same pattern as for policy question Q02.
RIGHT_LIMIT = r"\$\s?500(?:\.00)?(?!\d|,\d|\.\d)"     # "$500", but not "$5,000" or "$500.50"
WRONG_LIMIT = r"\$\s?1,?000(?:\.00)?(?!\d|,\d|\.\d)"   # "$1,000" or "$1000"
LIMIT_QUESTIONS = [
    {"id": "L1", "question": "So can I take out $800 in cash today?"},
    {"id": "L2", "question": "Sorry, what was my daily limit again?"},
    {"id": "L3", "question": "I need $900 in cash tomorrow. Can I withdraw it all in one day?"},
    {"id": "L4", "question": "My partner is on the same plan. Does the same daily limit apply to her, and how much is it?"},
    {"id": "L5", "question": "Is that limit per machine or for the whole day, and how much is it exactly?"},
]
for question in LIMIT_QUESTIONS:
    # "section": the policy section that holds the answer; "unusual": the answer is not unusual for a bank
    question.update({"answer": "$500 per day.", "check": [RIGHT_LIMIT], "section": 3, "unusual": False})
POISON_TEST = QUESTIONS + LIMIT_QUESTIONS   # the 25 questions of this exercise


def follow_ups_right(run):
    """How many of the five follow-up questions (ids L1 to L5) a run answered right."""
    return sum(row["right"] for row in run["rows"] if row["id"].startswith("L"))   # True counts as 1


def repeats_wrong_limit(row):
    """True when an answer states $1,000 and is not right (so "$500, not $1,000" does not count)."""
    return bool(re.search(WRONG_LIMIT, row["text"])) and not row["right"]


# ==========================================================================
# Step 3: Broken: run the 25 questions with the poisoned history
# ==========================================================================
def build_poisoned_context(question):
    """The usual context, but the history contains the poisoned turn."""
    return build_context(question, history=turns_as_messages(RAW_TURNS))


print()
print("=" * 70)
print("📊 BROKEN: THE HISTORY CONTAINS THE WRONG LIMIT")
print("=" * 70)
poisoned_run = score(build_poisoned_context, questions=POISON_TEST, title="poisoned history", model=SMALL_MODEL)
poisoned_repeats = [row for row in poisoned_run["rows"] if repeats_wrong_limit(row)]
print(f"Answers that repeat $1,000: {len(poisoned_repeats)} of {poisoned_run['total']}")
for row in poisoned_repeats:
    print(f"   {row['id']}: {row['text'][:140]}")

# ==========================================================================
# Step 4: The fix, part 1: check the amounts of a reply against the policy
# ==========================================================================
# We collect every dollar amount in a text as a number: "$1,000" becomes 1000.0, "$2.50" becomes 2.5.
def amounts_in(text):
    """Return the set of dollar amounts in a text, as numbers."""
    found = re.findall(r"\$\s?(\d[\d,]*(?:\.\d+)?)", text)    # the digits after a dollar sign, with commas and decimals
    return {float(amount.replace(",", "")) for amount in found}


POLICY_AMOUNTS = amounts_in(POLICY_TEXT)   # every amount the fee policy states


def reply_is_supported(reply):
    """True when every dollar amount in the reply also appears in the fee policy."""
    return all(amount in POLICY_AMOUNTS for amount in amounts_in(reply))


# ==========================================================================
# Step 5: The fix, part 2: only checked replies are saved to the history
# ==========================================================================
# In a live chat this check runs at the moment the reply is written, and a rejected reply is written again.
# Here the turns already exist, so a turn whose reply fails the check is left out.
def checked_turns(turns, report=True):
    """Return the turns whose bot reply passes the check. Print the ones that are left out (unless report=False)."""
    kept = []
    for customer_text, bot_reply in turns:
        if reply_is_supported(bot_reply):
            kept.append((customer_text, bot_reply))
        elif report:
            print(f"   left out of the history: '{bot_reply}' (it states an amount that is not in the policy)")
    return kept


print()
print("=" * 70)
print("📊 FIXED: EVERY REPLY IS CHECKED BEFORE IT IS SAVED")
print("=" * 70)
clean_turns = checked_turns(RAW_TURNS)


def build_checked_context(question):
    """The usual context, with a history that only holds checked replies."""
    return build_context(question, history=turns_as_messages(clean_turns))


checked_run = score(build_checked_context, questions=POISON_TEST, title="checked history", model=SMALL_MODEL)
checked_repeats = [row for row in checked_run["rows"] if repeats_wrong_limit(row)]

# ==========================================================================
# Step 6: Compare the two runs and store the result
# ==========================================================================
print()
print("=" * 70)
print("📊 POISONED AGAINST CHECKED, ON THE SAME 25 QUESTIONS")
print("=" * 70)
print(f"{'':<20}{'right answers':>15}{'right follow-ups':>18}{'repeat $1,000':>16}{'tokens in':>12}")
print(f"{'poisoned history':<20}{poisoned_run['right']:>9} of {poisoned_run['total']}{follow_ups_right(poisoned_run):>13} of 5"
      f"{len(poisoned_repeats):>16}{poisoned_run['tokens_in']:>12,}")
print(f"{'checked history':<20}{checked_run['right']:>9} of {checked_run['total']}{follow_ups_right(checked_run):>13} of 5"
      f"{len(checked_repeats):>16}{checked_run['tokens_in']:>12,}")
print("\nWrong answers with the poisoned history:")
show_wrong_answers(poisoned_run)
show_bill(poisoned_run["calls"] + checked_run["calls"], "Exercise 4")
record_result("poisoning", broken=poisoned_run, fixed=checked_run, fix="check replies before saving them",
              counted="follow-up questions answered wrongly (of 5)",
              counted_broken=5 - follow_ups_right(poisoned_run), counted_fixed=5 - follow_ups_right(checked_run))

# ==========================================================================
# Step 7: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
poisoned_history = build_poisoned_context("test")[1:-1]
print("✅ The poisoned history contains the bot reply with $1,000"
      if any(message["role"] == "assistant" and re.search(WRONG_LIMIT, message["content"]) for message in poisoned_history)
      else "❌ The poisoned history should contain the bot reply 'Your daily limit is $1,000.'")
print("✅ Your check rejects 'Your daily limit is $1,000.' and accepts a reply with $500"
      if not reply_is_supported("Your daily limit is $1,000.") and reply_is_supported("The daily limit is $500 on Standard.")
      and reply_is_supported("No. Electronic statements are free.")
      else "❌ reply_is_supported should be False for $1,000 and True for $500 and for a reply without amounts")
checked_history = build_checked_context("test")[1:-1]
print("✅ The checked history holds the two clean turns and no $1,000"
      if checked_history == turns_as_messages(EARLIER_TURNS)
      else "❌ The checked history should be exactly the two clean turns from Setup")
print("✅ Both runs have 25 answers, and the result is stored in RESULTS"
      if poisoned_run["total"] == 25 and checked_run["total"] == 25 and any(row["problem"] == "poisoning" for row in RESULTS)
      else "❌ Expected 25 answers in each run and a 'poisoning' row in RESULTS")
if follow_ups_right(poisoned_run) < follow_ups_right(checked_run):
    print(f"🤔 With the poisoned history the bot got {follow_ups_right(poisoned_run)} of the 5 follow-up questions right "
          f"({len(poisoned_repeats)} of all answers stated $1,000). With the checked history: {follow_ups_right(checked_run)} of 5. "
          "The wrong number can only be followed if it is allowed into the history.")
else:
    print(f"🤔 The bot got {follow_ups_right(poisoned_run)} of the 5 follow-up questions right with the poisoned history and "
          f"{follow_ups_right(checked_run)} of 5 with the checked one: in your run it did not follow the wrong limit more "
          "often than it erred anyway. That is the model resisting this failure, and it can go the other way on another "
          "day or with a longer history. The check makes the outcome certain: the wrong number is never in the context.")
print("🧠 A model cannot tell which of its earlier replies were wrong. Check a reply before it becomes history.")

# 🌫️ Topic 5: Distraction: The Right Paragraph Is Lost in Too Much Text

A tempting way to build a bot is to give it everything: "the model can read a million tokens, so paste in all the bank's documents and let it find the answer." The second way a context goes wrong is called **context distraction**: with too much text in the context, the model can answer from the wrong part of it.

Here is what that looks like. A customer on the Standard plan asks what a withdrawal at another bank's ATM costs. The bank's full terms and conditions contain the right paragraph. They also contain other paragraphs that answer almost the same question:

| Section of the terms and conditions | What it says about a withdrawal at another bank's ATM |
|---|---|
| Personal accounts: Cash withdrawals at ATMs | \$3 per withdrawal on Standard (the right answer) |
| Fee schedule 2024 (replaced): the same topic | the amount that applied until February 2026 |
| Business accounts: the same topic | the amount for a business plan |
| Credit cards: cash advances at ATMs | the amount for a cash advance |

Each of these paragraphs is true for somebody. Only one is true for this customer. A person who knows the bank skips the other three without thinking. A model reads all of them with the same attention, and the more look-alike paragraphs there are, the more likely it is to pick one of them. Researchers also describe a related effect under the name **context rot**: as the context gets longer, a model uses what is in it less reliably, and the middle suffers more than the start and the end.

So more context is not safer. It has three costs: the bill (every token is paid for in every call), the waiting time, and, as the text grows, the accuracy. There are two ways out, and you will build both:

- **Select:** give the model only the sections that match the customer and the question. This is what `relevant_sections` did in Exercise 1. Here it gets one more step, because the terms and conditions also hold sections for other products: first keep the sections that apply to this customer (personal accounts), then pick the ones that match the question.
- **Quote first:** keep the long document, put the question after it, and ask the model to quote the sentence that supports its answer before it answers. Writing the quote makes the model go and find the passage, and it gives you something to check: your code can test whether the quoted sentence really is in the document.

---

### 🔍 What You'll Learn

- Measure what a long context costs per question, in tokens and in money
- Select context in two steps: by who the customer is, then by what they asked
- Ask for a supporting quote, and verify the quote with code

### 🧠 Why This Matters

- "Just give it all the documents" is the most common first design of a document bot, and its mistakes are the hardest to notice: the answer is fluent and comes with a real number from a real section
- Selection is always cheaper, and often more accurate too. When you cannot select, a verified quote tells you which answers to trust

## 🧪 Exercise 5: Give the Bot the Full Terms and Conditions, Then Select Sections and Ask for Quotes

You replace the two policy sections in the bot's context with the bank's complete terms and conditions: one long document in which the 12 sections of the fee policy are spread among many others, including look-alike sections for business accounts, credit cards, savings accounts, branches in Canada and the fee schedule of 2024.

**The comparison.** You run the 20 policy questions three times and compare what each context costs per question, in input tokens and in money:

| Run | What is in the context |
|---|---|
| Full document | the full terms and conditions |
| Select | only the two sections for personal accounts that match the question |
| Quote first | the full terms and conditions, and after the question the instruction to quote the supporting sentence before answering |

The cost is what this exercise measures: the same 20 questions, with about fifty times the input in two of the three runs. For the third run your code also checks every quote against the document, so you see how many answers come with a sentence that really is in it.

Right answers are not compared here, and the reason is worth knowing. With this document and these questions, the small model finds the right paragraph nearly every time, with or without the sixty pages around it. Too much context starts to cost right answers with much longer inputs, with look-alike sections that a question cannot be told apart from, and with weaker models. Selecting sections and asking for a quote are what you do about it then, and you build both here.

This exercise runs on the small model (`SMALL_MODEL`), where a long input is a realistic worry and the cost per token is easy to follow.

### 🎯 What You'll Do

1. Load the terms and conditions, cut them into sections, and look at their size and their labels.
2. Run the 20 questions with the full document in the context, and read what one question costs.
3. Select: keep only the sections for personal accounts, pick the two that match the question, and run again.
4. Quote first: write the quote instruction, run again with the full document, and check the quotes with code.
5. Compare the three runs: tokens and cost per question, and the cost of 10,000 questions.

👉 By the end, you'll know what sixty pages in every call cost, and you'll have the two tools for a context that is too long.

---

### ✅ You Succeed When

- The full document stays below 100,000 tokens, where the larger model's price rises
- The first context carries the full document
- With selection, the context holds two sections, both for personal accounts, and for all 20 questions the section with the answer is one of them
- With quote first, the last message ends with your quote instruction
- The full document takes more input tokens per question than the two selected sections
- All three runs have 20 answers, and the result is stored in `RESULTS`

---

### 🛠 Technical Info

- **The sections:** `TERMS_SECTIONS` is a list of dictionaries: `number`, `label` (for example `"Personal accounts"` or `"Business accounts"`), `title` and `text`
- **Filtering a list:** `[item for item in items if condition]`
- **Picking sections from another list:** `relevant_sections(question, sections=...)` from Exercise 1
- **The small model:** `score(build, title=..., model=SMALL_MODEL)`
- **What one question costs:** `cost_per_question(run)` returns the input tokens and the dollars per question of a run
- **Honest note about the cost:** the bill of the two long runs is lower than tokens times price, because the provider of this model reads a beginning it has seen before from its cache (the idea of Exercise 3). The table shows both: what was billed, and the full price without a cache.
- **Honest note about the quotes:** the check compares each quote with the document after removing line breaks, double spaces and markdown stars. A model sometimes shortens a sentence when it quotes it, and such a quote counts as "not found".
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Run the selection without the filter on the label (pick the two best sections from all of `TERMS_SECTIONS`). For how many questions is a look-alike section now among the two?
2. Combine both: two selected sections and the quote instruction. Are more quotes found word for word when the context is short?

In [ ]:
# ==========================================================================
# 🌫️ Exercise 5: Give the Bot the Full Terms and Conditions, Then Select Sections and Ask for Quotes
# ==========================================================================

# Make sure the Setup cell and Exercises 1 and 2 ran first
assert "conversation_questions" in globals(), "Run the Setup cell and Exercises 1 and 2 first."

# ==========================================================================
# Step 1: Load the terms and conditions and the record of look-alike numbers (upload them if they are not here yet)
# ==========================================================================
need_file("bank_terms_and_conditions.md")
need_file("bank_terms_wrong_sources.json")

with open("bank_terms_and_conditions.md") as f:
    TERMS_TEXT = f.read()
with open("bank_terms_wrong_sources.json") as f:
    WRONG_SOURCES = json.load(f)["questions"]   # for each question id: the right section and the look-alike sections


def parse_sections(document_text):
    """Cut a document into its sections. A section starts with a heading like '## 7.2 Personal accounts: Cash withdrawals'."""
    sections = []
    for chunk in re.split(r"(?m)^(?=#{1,2} )", document_text):   # cut in front of every line that starts with "# " or "## "
        heading = re.match(r"## (\d+\.\d+) ([^:\n]+): (.+)", chunk)
        if heading:   # chunks that start with "# Part ..." or the title are not sections
            sections.append({"number": heading.group(1), "label": heading.group(2), "title": heading.group(3).strip(),
                             "text": chunk.strip()})
    return sections


TERMS_SECTIONS = parse_sections(TERMS_TEXT)
TERMS_TOKENS = count_tokens(TERMS_TEXT)

# ==========================================================================
# Step 2: Look at the document before using it
# ==========================================================================
print()
print("=" * 70)
print("📊 THE TERMS AND CONDITIONS")
print("=" * 70)
print(f"{len(TERMS_TEXT.split()):,} words · about {TERMS_TOKENS:,} tokens · {len(TERMS_SECTIONS)} sections")
print(f"(the two policy sections of the baseline context: about {BASELINE['tokens_in'] // BASELINE['total']:,} tokens per call)")
print("\nSections per label:")
labels = pd.Series([section["label"] for section in TERMS_SECTIONS]).value_counts()   # how often each label appears
for label, how_many in labels.items():
    print(f"   {label:<32}{how_many:>4}")
print("\nThe sections that could answer question Q01 (a withdrawal at another bank's ATM, Standard plan):")
print(f"   right : {WRONG_SOURCES['Q01']['right_section']}")
for source in WRONG_SOURCES["Q01"]["wrong_sources"]:
    print(f"   wrong : {source['section']}  (says {source['value']})")


# ==========================================================================
# Step 3: Two provided helpers
# ==========================================================================
def cost_per_question(run):
    """What one question costs in a run, on average: (input tokens, dollars billed)."""
    return run["tokens_in"] // run["total"], run["cost"] / run["total"]


def wrong_source_of(row):
    """For an answer that is not right: the look-alike section whose number it states, or None (used in Exercise 9)."""
    if row["right"]:
        return None
    for source in WRONG_SOURCES.get(row["id"], {}).get("wrong_sources", []):
        if all(re.search(pattern, row["text"].replace("*", ""), re.IGNORECASE) for pattern in source["check"]):
            return source["section"]
    return None


# ==========================================================================
# Step 4: The full document in the context
# ==========================================================================
# The whole terms and conditions take the place of the two policy sections.
def build_full_document_context(question):
    """The context with the complete terms and conditions as its only document."""
    return assemble(SYSTEM_PROMPT, [TERMS_TEXT], turns_as_messages(EARLIER_TURNS), question)


print()
print("=" * 70)
print("📊 THE FULL TERMS AND CONDITIONS IN EVERY CALL")
print("=" * 70)
full_document_run = score(build_full_document_context, title="full document", model=SMALL_MODEL, show_right=False)
tokens, dollars = cost_per_question(full_document_run)
print(f"One question: about {tokens:,} tokens in · ${dollars:.5f} billed")

# ==========================================================================
# Step 5: Select the sections that match the customer and the question
# ==========================================================================
# Two steps: first the sections that apply to this customer (personal accounts), then the two that match the question.
PERSONAL_SECTIONS = [section for section in TERMS_SECTIONS if section["label"] == "Personal accounts"]


def build_selected_context(question):
    """The context with the two personal-account sections of the terms that match the question."""
    sections = relevant_sections(question, sections=PERSONAL_SECTIONS)
    return assemble(SYSTEM_PROMPT, sections, turns_as_messages(EARLIER_TURNS), question)


print()
print("=" * 70)
print("📊 SELECT: ONLY THE SECTIONS THAT MATCH")
print("=" * 70)
print(f"Sections for personal accounts: {len(PERSONAL_SECTIONS)} of {len(TERMS_SECTIONS)}")
selected_run = score(build_selected_context, title="select", model=SMALL_MODEL, show_right=False)
tokens, dollars = cost_per_question(selected_run)
print(f"One question: about {tokens:,} tokens in · ${dollars:.5f} billed")

# ==========================================================================
# Step 6: Quote first: keep the full document and ask for a quote before the answer
# ==========================================================================
# The instruction goes after the question, at the very end of the context, where the model reads it last.
QUOTE_RULE = ("Before you answer, quote the one sentence from the documents that supports your answer, word for word, "
              "inside <quote> tags. Then give your answer.")


def build_quote_context(question):
    """The context with the full document, and the quote instruction after the question."""
    return assemble(SYSTEM_PROMPT, [TERMS_TEXT], turns_as_messages(EARLIER_TURNS), question + "\n\n" + QUOTE_RULE)


def plain(text):
    """A text without markdown stars and with single spaces, so quotes can be compared with the document."""
    return " ".join(text.replace("*", "").split())


def quote_is_in_document(reply, document_text):
    """True when the reply has a <quote> and every quote is found word for word in the document."""
    quotes = re.findall(r"<quote>(.*?)</quote>", reply, re.DOTALL)   # re.DOTALL: a quote may run over several lines
    return bool(quotes) and all(plain(quote).strip(' ".') in plain(document_text) for quote in quotes)


print()
print("=" * 70)
print("📊 QUOTE FIRST: THE FULL DOCUMENT, WITH A QUOTE BEFORE THE ANSWER")
print("=" * 70)
quote_run = score(build_quote_context, title="quote first", model=SMALL_MODEL, show_right=False)
tokens, dollars = cost_per_question(quote_run)
print(f"One question: about {tokens:,} tokens in · ${dollars:.5f} billed")
verified_quotes = sum(quote_is_in_document(row["text"], TERMS_TEXT) for row in quote_run["rows"])
print(f"Answers with a quote that is found word for word in the document: {verified_quotes} of {quote_run['total']}")
print("One answer of this run:")
print(textwrap.indent(textwrap.fill(quote_run["rows"][0]["text"], width=100), "   "))

# ==========================================================================
# Step 7: Compare what the three contexts cost, and store the result
# ==========================================================================
# "billed" is what the calls really cost. "full price" is input tokens times the price per token, with no cache.
print()
print("=" * 70)
print("📊 WHAT ONE QUESTION COSTS, AND 10,000 QUESTIONS")
print("=" * 70)
print(f"{'':<16}{'tokens in per question':>24}{'billed per question':>21}{'10,000 questions, billed':>26}{'at the full price':>19}")
for run in (full_document_run, selected_run, quote_run):
    tokens, dollars = cost_per_question(run)
    full_price = tokens * SMALL_PRICE_IN / 1_000_000   # the price is per MILLION tokens
    print(f"{run['title']:<16}{tokens:>24,}{dollars:>21.5f}{dollars * 10_000:>26.2f}{full_price * 10_000:>19.2f}")
tokens_ratio = full_document_run["tokens_in"] // max(1, selected_run["tokens_in"])
print(f"\nThe full document takes about {tokens_ratio} times the input tokens of the two selected sections, for the same question.")
print(f"Quote first: {verified_quotes} of {quote_run['total']} answers carry a quote that is found word for word in the document.")
show_bill(full_document_run["calls"] + selected_run["calls"] + quote_run["calls"], "Exercise 5")
record_result("distraction", broken=full_document_run, fixed=selected_run, fix="select the sections that match",
              counted="input tokens per question", compare_scores=False,
              counted_broken=full_document_run["tokens_in"] // 20, counted_fixed=selected_run["tokens_in"] // 20)

# ==========================================================================
# Step 8: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print(f"✅ The full document is about {TERMS_TOKENS:,} tokens, below 100,000" if TERMS_TOKENS < 100_000
      else f"❌ The document is about {TERMS_TOKENS:,} tokens: above 100,000 the larger model's price is five times higher")
print("✅ The first context carries the full document"
      if TERMS_TEXT in build_full_document_context("test")[0]["content"]
      else "❌ build_full_document_context should put TERMS_TEXT into the context as its document")
selection_misses = []
for question in QUESTIONS:
    system_text = build_selected_context(question["question"])[0]["content"]
    right_title = next(section["title"] for section in POLICY_SECTIONS if section["number"] == question["section"])
    headings = re.findall(r"(?m)^## \d+\.\d+ ([^:\n]+): (.+)$", system_text)   # (label, title) of every section in the context
    if len(headings) != 2 or any(label != "Personal accounts" for label, _ in headings) or \
            right_title not in [title.strip() for _, title in headings]:
        selection_misses.append(question["id"])
print("✅ Selection: two sections for personal accounts per question, and the section with the answer is always one of them"
      if not selection_misses and len(PERSONAL_SECTIONS) == 12
      else f"❌ Selection: wrong or missing sections for {', '.join(selection_misses) or 'no question'}; "
           f"PERSONAL_SECTIONS should hold the 12 sections with the label 'Personal accounts' (it holds {len(PERSONAL_SECTIONS)})")
print("✅ Quote first: the last message ends with your quote instruction"
      if build_quote_context("test")[-1]["content"].endswith(QUOTE_RULE) and "<quote>" in QUOTE_RULE
      and TERMS_TEXT in build_quote_context("test")[0]["content"]
      else "❌ Quote first: keep the full document, and end the last message with QUOTE_RULE (it should ask for <quote> tags)")
print("✅ All three runs have 20 answers, and the result is stored in RESULTS"
      if all(run["total"] == 20 for run in (full_document_run, selected_run, quote_run))
      and any(row["problem"] == "distraction" for row in RESULTS)
      else "❌ Expected 20 answers in each run and a 'distraction' row in RESULTS")
print(f"✅ The full document takes more input tokens per question than the two selected sections (about {tokens_ratio} times)"
      if full_document_run["tokens_in"] > selected_run["tokens_in"]
      else "❌ The full document should take more input tokens than two selected sections: check the two context builders")
if any(row["error"] for run in (full_document_run, selected_run, quote_run) for row in run["rows"]):
    print("🤔 Some calls failed and have no answer (see the bill above). Run the cell again.")
print(f"🤔 {verified_quotes} of {quote_run['total']} answers came with a quote that code found in the document. An answer whose "
      "quote is not found is one to look at before you trust it: the model may have shortened the sentence, or made it up.")
print("🧠 Every token of context is paid for in every call. Give the model the paragraphs that apply, and make it show where an answer comes from.")

# 🧰 Topic 6: Confusion: Too Many Tools to Choose From

So far the bot answered from text in its context. A bot can also be given **tools**: functions of the bank's systems that it may ask your code to run. The model cannot run anything itself. It sees a list of tool descriptions, and when it wants one, it replies with the tool's name and inputs. Your code runs the function and sends the result back in the next call. One tool description looks like this:

```python
{"name": "get_card_and_atm_fee_policy",
 "description": "Current fee policy for personal accounts, part 2: cash withdrawals at ATMs and the daily ATM "
                "withdrawal limit, card purchases in a foreign currency, replacement cards ...",
 "parameters": {}}
```

Tool descriptions are part of the context. They are sent with every call, the model reads all of them, and you pay for all of them. That leads to the third way a context goes wrong, called **context confusion**: when the model is offered many tools it does not need, it picks a wrong one more often.

Imagine the bank's catalog has 40 tools, and a customer asks what a paper statement costs. Three of the tools return parts of the current fee policy, and one of those three holds the answer. But the catalog also has `get_statement` (one customer's monthly statement), `order_paper_statement` (which changes the customer's settings), `get_fee_schedule` (an older schedule) and `search_help_center`. Every one of them has "statement" or "fee" in its description. The more such neighbours a tool has, the more often the model reaches for one of them. In the best case the wrong tool returns nothing useful and the model tries again, which costs a call. In the worst case it returns a number from the wrong product or the wrong year, and the bot passes it on.

The fix is the one from Topic 5, applied to tools: **select**. Before each call, your code compares the question with the tool descriptions and passes only the few tools that match. The model then chooses among three tools, not forty, and the descriptions cost a fraction of the tokens.

---

### 🔍 What You'll Learn

- See tool descriptions as part of the context: they cost tokens and compete for the model's attention
- Count the calls to a wrong tool with a check that code can do
- Select the tools for a question by matching it against the descriptions

### 🧠 Why This Matters

- Tool catalogs grow: every team adds "just one more", and connected services bring dozens at once. The cost and the wrong picks grow with them, and nobody changed the prompt
- A wrong tool call is more than a wrong word: tools that change something (freeze a card, send a wire) act on the real world

## 🧪 Exercise 6: Offer the Bot 40 Tools, Then Only the Ones That Match the Question

In this exercise the fee policy is **not** in the context. The bot has to look it up with a tool. The bank's catalog has 40 tools. Three of them return the current fee policy for personal accounts, one part each, and every one of the 20 questions is answered by exactly one of these three. The other 37 do other jobs. Six of them return information that looks similar but belongs to another product, another country or an older fee schedule.

**The comparison.** You run the 20 policy questions twice. First the bot is offered all 40 tools with every question. Then your code picks the three tools whose descriptions match the question best and offers only those. For both runs you count the calls to a wrong tool, the right answers, and the tokens the tool descriptions take up.

"A wrong tool" has an exact meaning here, so code can count it: each question has one right tool, the policy tool whose part holds the section with the answer. A call to any other tool is a wrong call.

### 🎯 What You'll Do

1. Look at the catalog: its size in tokens, and the right tool for a few questions.
2. Write the context for a bot that looks things up, and run the 20 questions with all 40 tools.
3. Count the calls to a wrong tool.
4. Write `select_tools(question)`: the three tools whose name and description match the question best.
5. Run the 20 questions again with the selected tools, and compare.

👉 By the end, you'll have a tool selector, and the count of wrong tool calls with 40 tools and with 3.

---

### ✅ You Succeed When

- The broken context offers all 40 tools
- `select_tools` returns 3 tools, and for all 20 questions the right tool is one of them
- The descriptions of the selected tools take fewer tokens than the full catalog
- Both runs have 20 answers, and the result is stored in `RESULTS`

---

### 🛠 Technical Info

- **A context with tools:** a dictionary `{"messages": [...], "tools": [...]}`. `score` and `ask_bot` accept it
- **What a run returns:** each row has `tools_called`, the names of the tools the bot called for that question, in order
- **The right tool:** `right_tool_for(question)` returns its name
- **A tool's text:** `tool["function"]["name"]` and `tool["function"]["description"]`
- **Matching:** `rank_by_word_overlap(question, texts)`, as for the policy sections
- **Honest note:** current models choose well among 40 clearly described tools, so the count of wrong calls may be small or zero in your run. The token count is certain: 40 descriptions are sent, and paid for, with every call. In real catalogs with hundreds of tools and uneven descriptions, the wrong picks rise.
- **Honest note about the fix:** selecting by description removes tools that have nothing to do with the question. A look-alike tool whose description matches the question well can still be among the three. Tools that return outdated information should be removed from the catalog itself, which is the subject of Exercise 7.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Set `TOOLS_PER_QUESTION` to 1 and run again. The description tokens fall further. Is the right tool still offered for all 20 questions?
2. Rewrite the description of one look-alike tool so that it says clearly what it must not be used for, run the 40-tool version again, and see whether the wrong calls to it change.

In [ ]:
# ==========================================================================
# 🧰 Exercise 6: Offer the Bot 40 Tools, Then Only the Ones That Match the Question
# ==========================================================================

# Make sure the Setup cell and Exercises 1 and 2 ran first
assert "conversation_questions" in globals(), "Run the Setup cell and Exercises 1 and 2 first."

# ==========================================================================
# Step 1: Look at the catalog
# ==========================================================================
def description_tokens(tools):
    """An estimate of the tokens a list of tool descriptions takes up in the context."""
    return count_tokens(json.dumps(tools))


print("=" * 70)
print("📊 THE TOOL CATALOG")
print("=" * 70)
print(f"{len(ALL_TOOLS)} tools · about {description_tokens(ALL_TOOLS):,} tokens of descriptions, sent with every call")
for tool in ALL_TOOLS[:5]:
    print(f"   {tool['function']['name']:<30}{tool['function']['description'][:75]}...")
print("   ...")
print("\nThe right tool for three of the questions:")
for question in (QUESTIONS[0], QUESTIONS[8], QUESTIONS[14]):
    print(f"   {question['id']}: {question['question'][:70]}...  ->  {right_tool_for(question)}")

# ==========================================================================
# Step 2: The context of a bot that looks the policy up
# ==========================================================================
# No documents this time: the system prompt tells the bot to use a tool, and the tools travel next to the messages.
TOOL_RULE = ("The fee policy is not in this conversation. Before you answer, look it up with one of your tools, "
             "and answer only from what the tool returns.")


def build_tool_context(question, tools):
    """The context for a bot that looks things up: no documents, and a list of tools next to the messages."""
    messages = assemble(SYSTEM_PROMPT + "\n" + TOOL_RULE, [], turns_as_messages(EARLIER_TURNS), question)
    return {"messages": messages, "tools": tools}


# ==========================================================================
# Step 3: Broken: every question gets all 40 tools
# ==========================================================================
def wrong_tool_calls(result):
    """Return (question id, tool name) for every call to a tool that is not the right tool of its question."""
    wrong = []
    for row in result["rows"]:
        question = next(question for question in QUESTIONS if question["id"] == row["id"])
        for name in row["tools_called"]:
            if name != right_tool_for(question):
                wrong.append((row["id"], name))
    return wrong


print()
print("=" * 70)
print("📊 BROKEN: ALL 40 TOOLS WITH EVERY QUESTION")
print("=" * 70)
all_tools_run = score(lambda question: build_tool_context(question, ALL_TOOLS), title="40 tools")
all_tools_wrong = wrong_tool_calls(all_tools_run)
print(f"Calls to a wrong tool: {len(all_tools_wrong)}")
for question_id, name in all_tools_wrong:
    print(f"   {question_id}: called {name}")
show_wrong_answers(all_tools_run)

# ==========================================================================
# Step 4: The fix: pass only the tools that match the question
# ==========================================================================
# The same matching as for policy sections: compare the question with each tool's name and description.
TOOLS_PER_QUESTION = 3


def select_tools(question, how_many=TOOLS_PER_QUESTION):
    """Return the tools whose name and description share the most meaningful words with the question."""
    texts = [tool["function"]["name"] + " " + tool["function"]["description"] for tool in ALL_TOOLS]
    order = rank_by_word_overlap(question, texts)   # positions in ALL_TOOLS, best match first
    return [ALL_TOOLS[position] for position in order[:how_many]]


print()
print("=" * 70)
print("📊 FIXED: ONLY THE TOOLS THAT MATCH THE QUESTION")
print("=" * 70)
for question in (QUESTIONS[0], QUESTIONS[14]):
    names = [tool["function"]["name"] for tool in select_tools(question["question"])]
    print(f"   {question['id']}: {question['question'][:60]}...\n        offered: {', '.join(names)}")
selected_tools_run = score(lambda question: build_tool_context(question, select_tools(question)), title="3 selected tools")
selected_tools_wrong = wrong_tool_calls(selected_tools_run)
print(f"Calls to a wrong tool: {len(selected_tools_wrong)}")
for question_id, name in selected_tools_wrong:
    print(f"   {question_id}: called {name}")

# ==========================================================================
# Step 5: Compare the two runs and store the result
# ==========================================================================
tokens_all = description_tokens(ALL_TOOLS)
tokens_selected = sum(description_tokens(select_tools(question["question"])) for question in QUESTIONS) // len(QUESTIONS)

print()
print("=" * 70)
print("📊 40 TOOLS AGAINST 3 SELECTED TOOLS, ON THE SAME 20 QUESTIONS")
print("=" * 70)
print(f"{'':<20}{'right':>10}{'wrong tool calls':>18}{'description tokens per call':>29}{'tokens in, all calls':>22}")
print(f"{'40 tools':<20}{all_tools_run['right']:>4} of {all_tools_run['total']}{len(all_tools_wrong):>18}"
      f"{tokens_all:>29,}{all_tools_run['tokens_in']:>22,}")
print(f"{'3 selected tools':<20}{selected_tools_run['right']:>4} of {selected_tools_run['total']}{len(selected_tools_wrong):>18}"
      f"{tokens_selected:>29,}{selected_tools_run['tokens_in']:>22,}")
show_bill(all_tools_run["calls"] + selected_tools_run["calls"], "Exercise 6")
record_result("confusion", broken=all_tools_run, fixed=selected_tools_run, fix="pass only the tools that match",
              counted="calls to a wrong tool", counted_broken=len(all_tools_wrong), counted_fixed=len(selected_tools_wrong))

# ==========================================================================
# Step 6: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ The broken context offers all 40 tools" if len(build_tool_context("test", ALL_TOOLS)["tools"]) == 40
      else "❌ The broken context should offer ALL_TOOLS (40 tools)")
not_offered = [question["id"] for question in QUESTIONS
               if right_tool_for(question) not in [tool["function"]["name"] for tool in select_tools(question["question"])]]
print("✅ select_tools returns 3 tools, and for all 20 questions the right tool is one of them"
      if not not_offered and all(len(select_tools(question["question"])) == 3 for question in QUESTIONS)
      else f"❌ select_tools should return 3 tools that include the right one; it is missing for: {', '.join(not_offered) or 'none'}")
print(f"✅ The selected tools take about {tokens_selected:,} description tokens per call; the full catalog takes {tokens_all:,}"
      if tokens_selected < tokens_all else "❌ The selected tools should take fewer description tokens than the full catalog")
print("✅ Both runs have 20 answers, and the result is stored in RESULTS"
      if all_tools_run["total"] == 20 and selected_tools_run["total"] == 20
      and any(row["problem"] == "confusion" for row in RESULTS)
      else "❌ Expected 20 answers in each run and a 'confusion' row in RESULTS")
never_called = [row["id"] for row in all_tools_run["rows"] + selected_tools_run["rows"] if not row["tools_called"]]
if never_called:
    print(f"🤔 Answers for which the bot called no tool at all and answered from memory or declined: {len(never_called)}. "
          "Such an answer counts as wrong when its number is not the policy's.")
if len(all_tools_wrong) > len(selected_tools_wrong):
    print(f"🤔 Wrong tool calls: {len(all_tools_wrong)} with 40 tools, {len(selected_tools_wrong)} with 3 selected tools. "
          "Fewer tools to choose from, fewer wrong picks, and a much smaller context.")
elif not all_tools_wrong:
    print("🤔 The bot called no wrong tool with 40 tools in your run: this model chooses well among clearly described "
          f"tools. The saving is still real: about {tokens_all - tokens_selected:,} tokens less in every call.")
else:
    print(f"🤔 Wrong tool calls: {len(all_tools_wrong)} with 40 tools, {len(selected_tools_wrong)} with 3 selected tools. "
          "Look at which tools were called by mistake: a look-alike tool that matches the question survives the "
          "selection. Such a tool has to leave the catalog, or its description has to say what it is not for.")
print("🧠 Tool descriptions are context too. Offer the model the tools for this question, not the whole catalog.")

# ⚔️ Topic 7: Clash: Two Sources in the Context Disagree

The fourth way a context goes wrong is called **context clash**: two pieces of the context give different answers to the same question, and nothing tells the model which one to believe.

It happens easily, because a bank has more than one text about its fees. The fee policy was updated in March 2026. A page of frequently asked questions (an FAQ page) in the help center was written in 2024 and never touched again. A search over "all our documents" finds both and puts both into the context:

| Source | What it says | Written |
|---|---|---|
| Fee policy, section 3 | *"Withdrawals at another bank's ATM inside the country: Standard: \$3 per withdrawal."* | March 2026 |
| Help center FAQ page | *"On the Standard plan, each withdrawal at another bank's ATM inside the country costs \$2.50."* | June 2024 |

A customer asks what the ATM fee is. Both sentences are in front of the model, both look official, and they contradict each other. The model has to pick, and without a reason to prefer one it may pick differently from one call to the next: \$3 for one customer, \$2.50 for the next. An answer that changes between two identical questions is the typical sign of a clash. A document that is out of date, like this FAQ page, is called a **stale document**.

There are two ways to settle a clash, and they differ in who does the deciding:

- **Tell the model which source wins.** Label every document with its date, and add a rule to the system prompt: when documents disagree, use the newest. This is called **source priority**. The model still reads both documents and applies your rule.
- **Remove the loser with code.** Before the context is built, your code drops every document that a newer document on the same topic has replaced. The model never sees the old number, so it cannot use it.

The second way is the stronger one whenever code can tell which document is outdated: a rule in a prompt is followed most of the time, and code runs every time. The first way is what you have left when both documents are needed, for example when the old page also holds information that the new one lacks.

---

### 🔍 What You'll Learn

- Recognise a clash by its sign: the same question gets different answers
- Give documents a date and a priority rule
- Remove outdated documents with code before they reach the context

### 🧠 Why This Matters

- Most companies have several versions of the same facts: an old PDF, a wiki page, last year's price list. A search over all of them brings the contradictions straight into the context
- A bot that tells two customers two different fees is a complaint waiting to happen, and each answer alone looks perfectly fine

## 🧪 Exercise 7: Put an Old FAQ Page Next to the Current Policy, Then Try Two Fixes

You put two sources into the bot's context: the policy sections that match the question (current, with an ATM fee of \$3) and an old FAQ page from the help center (written in 2024, with an ATM fee of \$2.50). The page also has three other numbers that the current policy changed: the price of a replacement card, the time limit for a dispute and the price of a paper statement.

**The comparison.** You measure three versions of the context, each in two ways:

| Version | What is in the context |
|---|---|
| Broken | the policy sections and the FAQ page, with nothing that says which is newer |
| Fix one: dates and a rule | both documents, each labelled with its name and date, and the rule "when documents disagree, use the newest" |
| Fix two: remove with code | your code drops the outdated document before the context is built |

The first measurement asks the ATM question ten times and counts how often \$3 and \$2.50 come back. The second runs all 20 policy questions, because the old page disagrees with the policy on four of them.

### 🎯 What You'll Do

1. Look at the two documents and where they disagree.
2. Build the broken context, ask the ATM question ten times, and count the two numbers.
3. Fix one: label each document with its date and write the rule that says which document wins.
4. Fix two: write `drop_outdated(documents)`, which keeps only the newest document of each topic.
5. Run all three versions on the ten ATM questions and on the 20 policy questions, and compare.

👉 By the end, you'll know how often the old number comes back, and which of the two fixes removes it for certain.

---

### ✅ You Succeed When

- The broken context contains both numbers, \$3 and \$2.50, and no dates
- With fix one, the context shows both dates and your rule
- `drop_outdated` keeps the fee policy and drops the FAQ page, and with fix two the context no longer contains \$2.50
- Each version has 10 answers to the ATM question and 20 answers to the policy questions, and the result is stored in `RESULTS`

---

### 🛠 Technical Info

- **A document here:** a dictionary with `name`, `date` (as `"YYYY-MM-DD"`, so plain text comparison puts dates in order), `topic` and `text`
- **The newest date:** `max(dates)` works on dates written as `"YYYY-MM-DD"`
- **Asking one question ten times:** `score(build, questions=[question] * 10, title=...)`
- **Honest note:** a model may prefer one of the two numbers consistently (ten times \$3, or ten times \$2.50), or name both and ask which applies. All of these are a clash: the answer is decided by something you did not choose. The count shows what your model does today.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Swap the order of the two documents in the broken context (FAQ page first, policy second) and ask ten times again. Does the number that comes back depend on which document the model read last?
2. In fix one, leave the rule out and keep only the date labels. Is the date alone enough for the model to choose the newer document?

In [ ]:
# ==========================================================================
# ⚔️ Exercise 7: Put an Old FAQ Page Next to the Current Policy, Then Try Two Fixes
# ==========================================================================

# Make sure the Setup cell and Exercises 1 and 2 ran first
assert "conversation_questions" in globals(), "Run the Setup cell and Exercises 1 and 2 first."

# ==========================================================================
# Step 1: The two documents, and where they disagree
# ==========================================================================
# A document is a dictionary: a name, the date it was written, its topic and its text.
FAQ_DOCUMENT = {"name": "Help center page: Fees FAQ", "date": "2024-06-12", "topic": "fees", "text": OLD_FAQ_TEXT}


def policy_document(question):
    """The current fee policy as a document: the two sections that match the question."""
    return {"name": "Personal Account Fee Policy", "date": "2026-03-01", "topic": "fees",
            "text": "\n\n".join(relevant_sections(question))}


# The four questions on which the old page disagrees with the policy, and the pattern of the old page's number
OLD_FAQ_VALUES = {
    "Q01": ("$2.50", r"\$\s?2\.50(?!\d)"),
    "Q03": ("30 days", r"\b(?:30|thirty)\s*(?:calendar\s+)?days?"),
    "Q13": ("$6", r"\$\s?6(?:\.00)?(?!\d|,\d|\.\d)"),
    "Q15": ("$3", r"\$\s?3(?:\.00)?(?!\d|,\d|\.\d)"),
}

print("=" * 70)
print("📊 WHERE THE OLD FAQ PAGE DISAGREES WITH THE CURRENT POLICY")
print("=" * 70)
for question in QUESTIONS:
    if question["id"] in OLD_FAQ_VALUES:
        print(f"   {question['id']}: {question['question'][:75]}")
        print(f"        policy (2026): {question['answer']:<60} FAQ page (2024): {OLD_FAQ_VALUES[question['id']][0]}")

ATM_QUESTION = QUESTIONS[0]   # "On the Standard plan, what does one cash withdrawal at another bank's ATM ... cost?"


def count_atm_numbers(result):
    """Sort the answers to the ATM question: the policy's $3, the old page's $2.50, both numbers, or neither."""
    counts = {"$3 (policy)": 0, "$2.50 (old page)": 0, "both numbers": 0, "neither": 0}
    for row in result["rows"]:
        says_new = is_correct(row["text"], ATM_QUESTION)
        says_old = bool(re.search(OLD_FAQ_VALUES["Q01"][1], row["text"]))
        if says_new and says_old:
            counts["both numbers"] += 1
        elif says_new:
            counts["$3 (policy)"] += 1
        elif says_old:
            counts["$2.50 (old page)"] += 1
        else:
            counts["neither"] += 1
    return counts


def old_values_given(result):
    """How many answers of a run on the 20 questions state the old page's number and are not right."""
    return sum(1 for row in result["rows"] if row["id"] in OLD_FAQ_VALUES and not row["right"]
               and re.search(OLD_FAQ_VALUES[row["id"]][1], row["text"], re.IGNORECASE))


# ==========================================================================
# Step 2: Broken: both documents in the context, with nothing that says which is newer
# ==========================================================================
def build_clash_context(question):
    """The context with the matching policy sections and the old FAQ page, as plain texts."""
    documents = [policy_document(question)["text"], FAQ_DOCUMENT["text"]]
    return assemble(SYSTEM_PROMPT, documents, turns_as_messages(EARLIER_TURNS), question)


# ==========================================================================
# Step 3: Fix one: label every document with its date, and say which one wins
# ==========================================================================
NEWEST_RULE = ("Each document shows the date it was written. When two documents disagree, "
               "use the document with the newest date and ignore the older one.")


def with_label(document):
    """The text of a document inside a tag that shows its name and its date."""
    return f'<document name="{document["name"]}" date="{document["date"]}">\n{document["text"]}\n</document>'


def build_dated_context(question):
    """The context with both documents, each labelled with its date, and the rule that the newest wins."""
    documents = [with_label(policy_document(question)), with_label(FAQ_DOCUMENT)]
    return assemble(SYSTEM_PROMPT + "\n" + NEWEST_RULE, documents, turns_as_messages(EARLIER_TURNS), question)


# ==========================================================================
# Step 4: Fix two: remove outdated documents with code
# ==========================================================================
# For every topic, only the document with the newest date stays. The model never sees the others.
def drop_outdated(documents):
    """Return the documents that are the newest of their topic."""
    kept = []
    for document in documents:
        newest_date = max(other["date"] for other in documents if other["topic"] == document["topic"])
        if document["date"] == newest_date:
            kept.append(document)
    return kept


def build_filtered_context(question):
    """The context with the documents that are left after the outdated ones were removed."""
    current = drop_outdated([policy_document(question), FAQ_DOCUMENT])
    return assemble(SYSTEM_PROMPT, [document["text"] for document in current], turns_as_messages(EARLIER_TURNS), question)


# ==========================================================================
# Step 5: Ask the ATM question ten times with each version
# ==========================================================================
versions = [("broken: two sources", build_clash_context), ("fix one: dates and a rule", build_dated_context),
            ("fix two: remove with code", build_filtered_context)]

print()
print("=" * 70)
print("📊 THE ATM QUESTION, TEN TIMES PER VERSION")
print("=" * 70)
print(f"Question: {ATM_QUESTION['question']}")
atm_runs = {}
for title, build in versions:
    atm_runs[title] = score(build, questions=[ATM_QUESTION] * 10, title=title)   # [x] * 10: a list with x ten times
print()
print(f"{'':<28}" + "".join(f"{label:>18}" for label in count_atm_numbers(atm_runs[versions[0][0]])))
for title, _ in versions:
    print(f"{title:<28}" + "".join(f"{how_many:>18}" for how_many in count_atm_numbers(atm_runs[title]).values()))

# ==========================================================================
# Step 6: Run the 20 policy questions with each version
# ==========================================================================
print()
print("=" * 70)
print("📊 THE 20 POLICY QUESTIONS PER VERSION")
print("=" * 70)
clash_runs = {title: score(build, title=title) for title, build in versions}
print()
print(f"{'':<28}{'right':>10}{'old page numbers given':>24}{'tokens in':>12}")
for title, _ in versions:
    run = clash_runs[title]
    print(f"{title:<28}{run['right']:>4} of {run['total']}{old_values_given(run):>24}{run['tokens_in']:>12,}")
print("\nWrong answers with the broken context:")
show_wrong_answers(clash_runs[versions[0][0]])
show_bill([call for run in list(atm_runs.values()) + list(clash_runs.values()) for call in run["calls"]], "Exercise 7")
record_result("clash", broken=clash_runs[versions[0][0]], fixed=clash_runs[versions[2][0]],
              fix="remove outdated documents with code",
              counted="ATM answers with the old number in them (of 10)",
              counted_broken=10 - count_atm_numbers(atm_runs[versions[0][0]])["$3 (policy)"],
              counted_fixed=10 - count_atm_numbers(atm_runs[versions[2][0]])["$3 (policy)"])

# ==========================================================================
# Step 7: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
clash_text = build_clash_context(ATM_QUESTION["question"])[0]["content"]
print("✅ The broken context contains both numbers, $3 and $2.50, and no dates"
      if "$3 per withdrawal" in clash_text and "$2.50" in clash_text and "2024-06-12" not in clash_text
      else "❌ The broken context should hold the policy sections and the FAQ page as plain texts, without dates")
dated_text = build_dated_context(ATM_QUESTION["question"])[0]["content"]
print("✅ Fix one: the context shows both dates and your rule"
      if "2026-03-01" in dated_text and "2024-06-12" in dated_text and NEWEST_RULE in dated_text and "$2.50" in dated_text
      else "❌ Fix one: label both documents with with_label() and add NEWEST_RULE to the system prompt")
kept_documents = drop_outdated([policy_document(ATM_QUESTION["question"]), FAQ_DOCUMENT])
other_topic = {"name": "Branch opening hours", "date": "2023-01-01", "topic": "branches", "text": "..."}
filtered_text = build_filtered_context(ATM_QUESTION["question"])[0]["content"]
print("✅ Fix two: drop_outdated keeps the fee policy and drops the FAQ page, and the context no longer contains $2.50"
      if [document["name"] for document in kept_documents] == ["Personal Account Fee Policy"] and "$2.50" not in filtered_text
      and other_topic in drop_outdated([FAQ_DOCUMENT, other_topic])
      else "❌ Fix two: drop_outdated should keep the newest document of each topic (and an old document on another topic stays)")
print("✅ Each version has 10 answers to the ATM question and 20 to the policy questions, and the result is stored"
      if all(run["total"] == 10 for run in atm_runs.values()) and all(run["total"] == 20 for run in clash_runs.values())
      and any(row["problem"] == "clash" for row in RESULTS)
      else "❌ Expected 10 and 20 answers per version and a 'clash' row in RESULTS")
as_text = lambda counts: ", ".join(f"{how_many} x {label}" for label, how_many in counts.items() if how_many)   # "4 x $3 (policy), ..."
broken_counts = count_atm_numbers(atm_runs[versions[0][0]])
filtered_counts = count_atm_numbers(atm_runs[versions[2][0]])
if broken_counts["$3 (policy)"] == 10:
    print("🤔 With both sources in the context, the bot said $3 all ten times in your run: it preferred the policy. "
          "Nothing in the context told it to, so this is the model's choice, not yours. Look at the 20 questions too: "
          f"{old_values_given(clash_runs[versions[0][0]])} answers there gave a number from the old page.")
else:
    print(f"🤔 With both sources in the context, ten identical questions got: {as_text(broken_counts)}. "
          "An answer that changes or mixes two numbers is the sign of a clash.")
print(f"🤔 After removing the old page with code: {as_text(filtered_counts)}. The old number cannot come back, because it is "
      "not in the context. With dates and a rule the model still reads it, and following the rule is up to the model.")
print("🧠 When two sources disagree, decide which one wins before the model has to. Code that removes the loser is the surest way.")

# 🙈 Topic 8: Ignored Context: The Model Answers From Memory

The four failures so far had one thing in common: something in the context was wrong, too much or contradictory. The fifth problem is different. The context is fine, and the model does not use it. This is called **ignored context**: the model answers from what it learned in training and passes over the document in front of it.

A model has read an enormous amount of text about banks. From that it has a strong idea of what is normal: a customer has 60 days to dispute a charge, an overdraft fee is about \$35, a foreign purchase costs about 3%. Ask it such a question with no document at all and it will often give you these typical values, stated as fact.

Now take a bank that does things differently. Its policy says:

> *"A dispute must be opened within 45 days of the date of the statement on which the charge first appears."*

The customer asks how long they have to dispute a charge. When the policy is in the context, right next to the question, a model reads it and says 45 days. The pull of "60 days" gets stronger the harder the document is to use: when the context is long and the sentence sits somewhere in the middle, when the fact is stated indirectly and has to be worked out, and when the document contradicts something the model has seen thousands of times. When the model gives in, the reply says 60 days, in a confident sentence, with the correct sentence somewhere above it in the context. Nobody notices until a customer disputes a charge on day 50.

The habit that protects you is called **grounding**: tying every answer to a source in the context, so that the answer can be checked. Two instructions do most of the work, and they are stronger together:

- **"Answer only from the documents provided."** This tells the model that its memory is not a source here.
- **"Quote the sentence you used."** To quote, the model has to find the sentence first, and the quote ends up in the reply, where code can compare it with the document.

---

### 🔍 What You'll Learn

- See what a model says from memory, by asking with no document in the context
- See that the same model uses the policy once it is in the context
- Ground the bot with an instruction to answer only from the documents and to quote, and check the quotes with code

### 🧠 Why This Matters

- Your company's documents matter most where they differ from what is usual: special prices, own deadlines, exceptions. That is exactly where a model's memory pulls hardest in the wrong direction
- An ungrounded wrong answer looks like every right answer. A quote gives you something that code can check

## 🧪 Exercise 8: Ask Five Questions With Unusual Answers, Then Ground the Bot

Five of the 20 policy questions have an answer that is unusual for a bank. This bank allows 45 days to dispute a charge where 60 days is common, charges \$20 for an overdraft item where \$35 is common, and so on. The data marks these five and gives the common value for each.

You ask the five questions three times, each time with a different context, and read what comes back:

| Run | What is in the context | What it shows |
|---|---|---|
| No document | the system prompt and the question only | what the model's memory holds |
| Policy in the context | the usual context from Exercise 1 | whether the bot uses the policy |
| Grounded | the same, with the instruction "answer only from the documents, and quote the sentence you used" | answers that carry their own evidence |

For the third run your code checks every quote against the policy text. That is the point of grounding in this exercise: an answer with a quote that code finds in the document can be trusted without a person reading it, and an answer without one can be sent to a person.

This exercise does not try to catch the bot ignoring the policy. With two short sections directly above the question, the model uses them, and you will see that in the second run. A model is more likely to answer from memory with a long context, with a fact that is stated indirectly, and with a value that contradicts something very well known. Grounding is the habit you want in place before that happens.

This exercise runs on the small model (`SMALL_MODEL`): with no document it states the values that are common at banks as if they were this bank's, which shows clearly what its memory holds.

### 🎯 What You'll Do

1. Look at the five unusual questions: the bank's value and the common value.
2. Ask them with no document in the context and read what the model says from memory.
3. Ask them with the usual context and see which value the bot gives now.
4. Add the grounding instruction to the system prompt and ask again.
5. Check with code how many answers of each run contain a quote that really is in the policy.

👉 By the end, you'll have seen what the model believes by itself, and you'll have a bot whose answers come with a quote that code can check.

---

### ✅ You Succeed When

- There are 5 unusual questions, and each of the three runs has 5 answers
- The context without documents holds no policy text
- The grounded context still holds the policy sections, and its system message contains your grounding instruction
- The result is stored in `RESULTS`

---

### 🛠 Technical Info

- **The unusual questions:** `question["unusual"]` is `True` for them; `question["common_value"]` is the typical value and `question["common_value_check"]` its patterns
- **No documents:** `assemble(SYSTEM_PROMPT, [], history, question)` builds a context without the documents layer
- **A longer system prompt:** add `GROUNDING_RULE` to `SYSTEM_PROMPT` on a new line, and pass the result to `assemble` as the system prompt
- **Honest note about the quotes:** a quote counts as found when it is in the policy word for word (after removing line breaks, double spaces and markdown stars). A model sometimes shortens a sentence or joins two when it quotes, and such a quote counts as not found. So expect some grounded answers without a checked quote: those are the ones a person would look at.
- **Difficulty:** 🟡 Level 2, Scaffolded

---

### 🚀 Level Up (optional)

1. Combine this with Exercise 5: ask the five unusual questions with the full terms and conditions in the context, with and without the grounding instruction, and compare the answers and the quotes.
2. Add a sixth instruction to the grounding rule: "If the documents do not contain the answer, say that you do not know." Then ask a question the policy does not answer, for example about mortgage rates, with and without it.

In [ ]:
# ==========================================================================
# 🙈 Exercise 8: Ask Five Questions With Unusual Answers, Then Ground the Bot
# ==========================================================================

# Make sure the Setup cell and Exercises 1 and 2 ran first
assert "conversation_questions" in globals(), "Run the Setup cell and Exercises 1 and 2 first."

# ==========================================================================
# Step 1: The five questions whose answer is unusual for a bank
# ==========================================================================
UNUSUAL_QUESTIONS = [question for question in QUESTIONS if question["unusual"]]

print("=" * 70)
print("📊 FIVE ANSWERS THAT ARE UNUSUAL FOR A BANK")
print("=" * 70)
print(f"{'id':<5}{'this bank':<52}{'common at banks':<18}")
for question in UNUSUAL_QUESTIONS:
    print(f"{question['id']:<5}{question['answer'][:50]:<52}{question['common_value']:<18}")


def gives_common_value(row):
    """True when an answer states the value that is common at banks and is not right for this bank."""
    question = next(question for question in UNUSUAL_QUESTIONS if question["id"] == row["id"])
    reply = row["text"].replace("*", "")   # without the stars of **bold** text
    states_common = all(re.search(pattern, reply, re.IGNORECASE) for pattern in question["common_value_check"])
    return states_common and not row["right"]


# ==========================================================================
# Step 2: What does the model say with no document at all?
# ==========================================================================
# No policy in the context, so whatever the bot answers comes from its training, or it says that it does not know.
def build_context_without_documents(question):
    """The context with the system prompt, the earlier turns and the question, and no documents."""
    return assemble(SYSTEM_PROMPT, [], turns_as_messages(EARLIER_TURNS), question)


print()
print("=" * 70)
print("📊 NO DOCUMENT IN THE CONTEXT: WHAT THE MODEL'S MEMORY HOLDS")
print("=" * 70)
memory_run = score(build_context_without_documents, questions=UNUSUAL_QUESTIONS, title="no documents", model=SMALL_MODEL)
for row in memory_run["rows"]:
    note = "the common value" if gives_common_value(row) else ("this bank's value" if row["right"] else "another answer")
    print(f"   {row['id']} ({note}): {row['text'].replace(chr(10), ' ')[:120]}")   # chr(10) is a line break

# ==========================================================================
# Step 3: The policy is in the context
# ==========================================================================
# build_context is the usual context from Exercises 1 and 2: system prompt, two policy sections, history, question.
def show_answers(run):
    """Print every answer of a run with a note: this bank's value, the common value, or another answer."""
    for row in run["rows"]:
        note = "the common value" if gives_common_value(row) else ("this bank's value" if row["right"] else "another answer")
        print(f"   {row['id']} ({note}): {row['text'].replace(chr(10), ' ')[:120]}")   # chr(10) is a line break


print()
print("=" * 70)
print("📊 THE POLICY IN THE CONTEXT")
print("=" * 70)
policy_run = score(build_context, questions=UNUSUAL_QUESTIONS, title="policy in the context", model=SMALL_MODEL)
show_answers(policy_run)

# ==========================================================================
# Step 4: Ground the bot: tell it to answer only from the documents, and to quote
# ==========================================================================
GROUNDING_RULE = "Answer only from the documents provided, and quote the sentence you used."


def build_grounded_context(question):
    """The usual context, with the grounding instruction added to the system prompt."""
    return assemble(SYSTEM_PROMPT + "\n" + GROUNDING_RULE, relevant_sections(question),
                    turns_as_messages(EARLIER_TURNS), question)


print()
print("=" * 70)
print("📊 GROUNDED: ANSWER ONLY FROM THE DOCUMENTS, AND QUOTE")
print("=" * 70)
grounded_run = score(build_grounded_context, questions=UNUSUAL_QUESTIONS, title="grounded", model=SMALL_MODEL)
for row in grounded_run["rows"]:
    print(f"   {row['id']}: " + textwrap.fill(row["text"].replace(chr(10), " "), width=100, subsequent_indent="        "))

# ==========================================================================
# Step 5: Check the quotes with code
# ==========================================================================
# A quote counts when at least one passage in quotation marks or <quote> tags (10 characters or more) is found in the policy text.
def has_quote_from(reply, document_text):
    """True when the reply contains a quoted passage that is found word for word in the document."""
    passages = re.findall(r"<quote>(.*?)</quote>", reply, re.DOTALL)   # text inside <quote> tags
    passages += re.findall(r'"([^"]{10,})"', reply)                    # text inside quotation marks
    document = " ".join(document_text.replace("*", "").split())
    return any(" ".join(passage.replace("*", "").split()).strip(" .") in document for passage in passages)


def summary(run):
    """For one run: answers with this bank's value, with the common value, and with a quote found in the policy."""
    return (run["right"], sum(gives_common_value(row) for row in run["rows"]),
            sum(has_quote_from(row["text"], POLICY_TEXT) for row in run["rows"]))


policy_quotes = summary(policy_run)[2]
grounded_quotes = summary(grounded_run)[2]

print()
print("=" * 70)
print("📊 THE THREE RUNS ON THE SAME FIVE QUESTIONS")
print("=" * 70)
print(f"{'':<24}{'value of this bank':>19}{'the common value':>18}{'quote found in policy':>23}")
for run in (memory_run, policy_run, grounded_run):
    bank_value, common_value, quotes = summary(run)
    print(f"{run['title']:<24}{bank_value:>14} of 5{common_value:>13} of 5{quotes:>18} of 5")
show_bill(memory_run["calls"] + policy_run["calls"] + grounded_run["calls"], "Exercise 8")
record_result("ignored context", broken=policy_run, fixed=grounded_run, fix="answer only from the documents, and quote",
              counted="answers with a quote found in the policy (of 5)", compare_scores=False,
              counted_broken=policy_quotes, counted_fixed=grounded_quotes)

# ==========================================================================
# Step 6: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
print("✅ There are 5 unusual questions, and each of the three runs has 5 answers"
      if len(UNUSUAL_QUESTIONS) == 5 and all(run["total"] == 5 for run in (memory_run, policy_run, grounded_run))
      else f"❌ Expected 5 unusual questions and 5 answers per run, got {len(UNUSUAL_QUESTIONS)} questions and "
           f"{memory_run['total']}, {policy_run['total']} and {grounded_run['total']} answers")
print("✅ The context without documents holds no policy text"
      if "<documents>" not in build_context_without_documents("test")[0]["content"]
      else "❌ build_context_without_documents should pass an empty list of documents to assemble()")
grounded_system = build_grounded_context(UNUSUAL_QUESTIONS[0]["question"])[0]["content"]
print("✅ The grounded context still holds the policy sections, and its system message contains your grounding instruction"
      if GROUNDING_RULE in grounded_system and "## 8. Disputing a charge" in grounded_system
      and "only" in GROUNDING_RULE.lower() and "quote" in GROUNDING_RULE.lower()
      else "❌ The grounded context should hold the two policy sections and GROUNDING_RULE (answer only from the documents, and quote)")
print("✅ The result is stored in RESULTS" if any(row["problem"] == "ignored context" for row in RESULTS)
      else "❌ The 'ignored context' row is missing in RESULTS")
if any(row["error"] for run in (memory_run, policy_run, grounded_run) for row in run["rows"]):
    print("🤔 Some calls failed and have no answer (see the bill above). Run the cell again.")
print(f"🤔 With no document, the bot gave the common value for {summary(memory_run)[1]} of the 5 questions: that is what its "
      "memory holds. (A bot that says it does not know is the better case.)")
print(f"🤔 With the policy in the context it gave this bank's value for {policy_run['right']} of 5, and with the grounding "
      f"instruction for {grounded_run['right']} of 5.")
print(f"🤔 Answers with a quote that code found in the policy: {policy_quotes} of 5 without the instruction, "
      f"{grounded_quotes} of 5 with it. A grounded answer without a checked quote is one for a person to look at.")
print("🧠 Grounding makes an answer checkable: tell the model to answer from the documents, and ask for the sentence it used.")

# 🏗️ Topic 9: One Context Builder With Every Fix in It

You have now seen five ways a context fails, and each fix turned out to be a few lines of ordinary code or one sentence in the system prompt. None of them needed a better model:

| Problem | What goes wrong | The fix | Where the fix lives |
|---|---|---|---|
| **Poisoning** | a wrong fact in the history is repeated | check a reply before it is saved | code |
| **Distraction** | the right paragraph is lost in too much text | select the sections that match the customer and the question | code |
| **Confusion** | too many tools, wrong picks | pass only the tools that match the question | code |
| **Clash** | two sources disagree | remove the outdated source (or date the sources and say which wins) | code (or the prompt) |
| **Ignored context** | the model answers from memory | "answer only from the documents, and quote" | the prompt |

In a real product these problems do not arrive one at a time. A support bot that has been running for a year has a long history, a document store with old and new versions, a tool catalog that every team added to, and a system prompt nobody has read lately. So the fixes belong together, in the one place where the context is built.

That place is a function like the `build_context()` you have carried through this notebook. It takes the **raw material** (every turn of the conversation, every document that the search found, every tool in the catalog) and decides what the model gets to read:

```python
def build_context(question):
    history   = ...   # only replies that passed the check
    documents = ...   # outdated ones removed, then the sections that match the customer and the question
    tools     = ...   # only the ones that match the question
    rules     = ...   # the system prompt with the grounding instruction
    return {"messages": assemble(rules, documents, history, question), "tools": tools}
```

Seen this way, the context builder is a filter between everything the bank has and the little the model should see. Each line of it answers one question: is this true, is this needed, is this current, and will the model use it?

---

### 🔍 What You'll Learn

- Combine five separate fixes in one function that builds the context
- Test the function against a context that has every problem at once
- Read a before-and-after table and say what each fix bought

### 🧠 Why This Matters

- A fix that lives in one notebook cell is forgotten. A fix that lives in the function every call goes through is applied every time
- With the context built in one place, a wrong answer can be traced: print the context of that call and look at each layer

## 🧪 Exercise 9: Put Every Fix Into `build_context()` and Run It Against a Context With All Five Problems

You write two functions that start from the same raw material: the three earlier turns (one of them with the wrong limit), two documents (the full terms and conditions and the old FAQ page), and the catalog of 40 tools.

`build_bad_context(question)` passes all of it on: the unchecked history, both documents in full, every tool, and the plain system prompt. That is a context with all five problems at once. `build_context(question)` is the final version of the function you started in Exercise 1: it applies every fix from Exercises 4 to 8 before anything reaches the model.

**The comparison.** You run the 25 questions of Exercise 4 (the 20 policy questions and the five follow-up questions about the limit) with each function, on the small model, and put the two runs next to each other: right answers, input tokens and cost. For every wrong answer of the bad context, a provided function names the likely cause. Then you print the table this notebook has been filling: one row per problem with the model, the score while broken, the score after the fix, the tokens used and the count that shows the problem most directly, and a last row for all five at once. For distraction and ignored context the table shows what those exercises measured (input tokens, quotes found) and no score.

This is the hardest exercise of the notebook: you write both functions from their descriptions, using the pieces you built before.

### 🎯 What You'll Do

1. Collect the raw material: turns, documents and tools.
2. Write `build_bad_context(question)`: everything goes in, unchecked.
3. Write the final `build_context(question)` with all five fixes.
4. Run the 25 questions with both, and look at the causes of the wrong answers.
5. Print the final table and the chart.

👉 By the end, you'll have one function that builds a clean context from messy material, and a table that shows what every fix was worth.

---

### ✅ You Succeed When

- The bad context has all five problems: the wrong limit in the history, both documents in full, 40 tools, and no grounding instruction
- The fixed context has none of them: no \$1,000 and no \$2.50 anywhere, two sections for personal accounts with the right one among them, three tools with the right one among them, and the grounding instruction
- The fixed context is at least 20 times smaller than the bad one
- Both runs have 25 answers, and the final table has 6 rows

---

### 🛠 Technical Info

- **Your pieces from earlier exercises:** `checked_turns(turns)` (4), `parse_sections(text)`, `relevant_sections(question, sections=...)` (5 and 1), `select_tools(question)` (6), `drop_outdated(documents)` (7), `GROUNDING_RULE` (8), `assemble(...)` and `turns_as_messages(...)` (1)
- **A context with tools:** `{"messages": messages, "tools": tools}`
- **The questions and the model:** `score(build, questions=POISON_TEST, title=..., model=SMALL_MODEL)`
- **Honest note:** in the fixed context the policy sections are in the documents, so the bot can answer without a tool. The three selected tools are still offered, as they would be in a real bot that also has other jobs.
- **Honest note about the result:** in our runs the small model got 21 to 23 of the 25 questions right with the bad context and 24 or 25 with the fixed one (it misreads one policy question in some runs, whatever the context). Most of the 20 policy questions survive the bad context, partly because the model calls the policy tools and reads the right sections there, and each such call sends the whole bad context again. The follow-up questions are where it fails: it allows withdrawals up to the wrong limit. The certain differences are the tokens and the guarantees: the fixed context is a small fraction of the size, and it cannot repeat the wrong limit or the old fee, because neither is in it.
- **Difficulty:** 🔴 Level 3, Independent

---

### 🚀 Level Up (optional)

1. Switch the fixes off one at a time (for example pass `RAW_TURNS` unchecked) and run again. Which single fix does the most for the score, and which for the tokens?
2. Add a sixth layer of defence: after the bot answers, check the amounts in its reply with `reply_is_supported()` from Exercise 4 before the reply is shown to the customer.

In [ ]:
# ==========================================================================
# 🏗️ Exercise 9: Put Every Fix Into build_context() and Run It Against a Context With All Five Problems
# ==========================================================================

# Make sure Exercises 4 to 8 ran first (each of them creates one of the fixes)
for name in ["checked_turns", "parse_sections", "select_tools", "drop_outdated", "GROUNDING_RULE"]:
    assert name in globals(), f"Run Exercises 4 to 8 first: '{name}' is missing."

# ==========================================================================
# Step 1: The raw material: what arrives before anything is checked or selected
# ==========================================================================
# RAW_TURNS comes from Exercise 4: two clean turns and the turn with the wrong limit.
# Two documents on the same topic: the current terms and conditions, and the old FAQ page.
RAW_DOCUMENTS = [
    {"name": "Terms and Conditions", "date": "2026-03-01", "topic": "fees", "text": TERMS_TEXT},
    FAQ_DOCUMENT,
]
print("=" * 70)
print("📊 THE RAW MATERIAL")
print("=" * 70)
print(f"Turns     : {len(RAW_TURNS)} (one of them states a wrong limit)")
for document in RAW_DOCUMENTS:
    print(f"Document  : {document['name']} ({document['date']}), about {count_tokens(document['text']):,} tokens")
print(f"Tools     : {len(ALL_TOOLS)}, about {description_tokens(ALL_TOOLS):,} tokens of descriptions")


# ==========================================================================
# Step 2: The bad context: everything goes in
# ==========================================================================
# Unchecked history, both documents in full, all 40 tools, the plain system prompt.
def build_bad_context(question):
    """A context with all five problems at once."""
    history = turns_as_messages(RAW_TURNS)
    documents = [document["text"] for document in RAW_DOCUMENTS]
    messages = assemble(SYSTEM_PROMPT, documents, history, question)
    return {"messages": messages, "tools": ALL_TOOLS}


# ==========================================================================
# Step 3: The final build_context: every fix, in one place
# ==========================================================================
# Each line applies one fix from Exercises 4 to 8 to the raw material.
# This function replaces the build_context of Exercise 2. To run Exercises 4 to 8 again later, run Exercise 2 again first.
def build_context(question):
    """The bot's context for one question, built from the raw material with every fix applied."""
    history = turns_as_messages(checked_turns(RAW_TURNS, report=False))   # poisoning: only checked replies
    current_documents = drop_outdated(RAW_DOCUMENTS)                      # clash: outdated documents removed
    personal_sections = [section for document in current_documents       # distraction, step 1: sections for this customer
                         for section in parse_sections(document["text"])
                         if section["label"] == "Personal accounts"]
    sections = relevant_sections(question, sections=personal_sections)    # distraction, step 2: sections for this question
    system_prompt = SYSTEM_PROMPT + "\n" + GROUNDING_RULE                 # ignored context: answer from the documents
    messages = assemble(system_prompt, sections, history, question)
    return {"messages": messages, "tools": select_tools(question)}        # confusion: only the matching tools


def context_tokens(context):
    """An estimate of the tokens of a whole context: all messages and all tool descriptions."""
    return count_tokens(json.dumps(context["messages"])) + description_tokens(context["tools"])


# ==========================================================================
# Step 4: Run the 25 questions of Exercise 4 with the bad context and with the fixed one
# ==========================================================================
print()
print("=" * 70)
print("📊 ALL FIVE PROBLEMS AT ONCE, AGAINST EVERY FIX")
print("=" * 70)
example_question = QUESTIONS[0]["question"]
print(f"One bad context  : about {context_tokens(build_bad_context(example_question)):,} tokens")
print(f"One fixed context: about {context_tokens(build_context(example_question)):,} tokens")
bad_run = score(build_bad_context, questions=POISON_TEST, title="bad context", model=SMALL_MODEL)
fixed_run = score(build_context, questions=POISON_TEST, title="fixed context", model=SMALL_MODEL)


# ==========================================================================
# Step 5: Why did the bad context give wrong answers?
# ==========================================================================
# For each wrong answer we test the causes this notebook taught, with the checks from Exercises 4 to 8.
def likely_causes(row):
    """Name the problems that fit a wrong answer: which wrong number it states, which wrong tool was called."""
    causes = []
    question = next(question for question in POISON_TEST if question["id"] == row["id"])
    if repeats_wrong_limit(row) or row["id"].startswith("L"):
        causes.append("poisoning (follows the wrong limit of $1,000)")
    if row["id"] in OLD_FAQ_VALUES and re.search(OLD_FAQ_VALUES[row["id"]][1], row["text"], re.IGNORECASE):
        causes.append("clash (the old FAQ page's number)")
    if wrong_source_of(row):
        causes.append(f"distraction (answered from {wrong_source_of(row)})")
    if question["unusual"] and gives_common_value(row):
        causes.append("ignored context (the common value)")
    if any(name != right_tool_for(question) for name in row["tools_called"]):
        causes.append("confusion (called " + ", ".join(row["tools_called"]) + ")")
    return causes or ["no known cause recognised"]


print("\nWrong answers with the bad context, and what fits them:")
for row in bad_run["rows"]:
    if not row["right"]:
        print(f"   {row['id']}: {row['text'].replace(chr(10), ' ')[:100] or '(no answer)'}")   # chr(10) is a line break
        print(f"        likely cause: {'; '.join(likely_causes(row))}")
if bad_run["right"] == bad_run["total"]:
    print("   (no wrong answers in this run)")
bad_tool_calls = sum(len(row["tools_called"]) for row in bad_run["rows"])
print(f"Tool calls made with the bad context: {bad_tool_calls} (each one is an extra model call that reads the whole context again)")
print("\nWrong answers with the fixed context:")
show_wrong_answers(fixed_run)
show_bill(bad_run["calls"] + fixed_run["calls"], "Exercise 9")
record_result("all five at once", broken=bad_run, fixed=fixed_run, fix="every fix, in build_context()",
              counted="follow-up questions answered wrongly (of 5)",
              counted_broken=5 - follow_ups_right(bad_run), counted_fixed=5 - follow_ups_right(fixed_run))

# ==========================================================================
# Step 6: The final table and the chart
# ==========================================================================
# One row per problem. First the scores and the input tokens of both runs, then the count that shows each problem
# most directly. A score can hide a problem that this count shows: read both.
final_table = pd.DataFrame(RESULTS)
print()
print("=" * 70)
print("📊 THE FINAL TABLE: EVERY PROBLEM, BROKEN AND FIXED")
print("=" * 70)
print(final_table[["problem", "model", "score broken", "score fixed", "tokens in, broken", "tokens in, fixed"]].to_string(index=False))
print()
print(final_table[["problem", "what was counted", "broken", "fixed", "the fix"]].to_string(index=False))

figure, axis = plt.subplots(figsize=(9, 4))
chart_table = final_table.dropna(subset=["share broken"])   # only the problems for which right answers were compared
positions = range(len(chart_table))
axis.bar([position - 0.2 for position in positions], chart_table["share broken"] * 100, width=0.4, color="lightgrey", label="broken")
axis.bar([position + 0.2 for position in positions], chart_table["share fixed"] * 100, width=0.4, color="tab:blue", label="fixed")
axis.set_xticks(list(positions))
axis.set_xticklabels(chart_table["problem"], rotation=20, ha="right")   # ha="right": the end of each label sits under its bars
axis.set_ylabel("Right answers (%)")
axis.set_ylim(0, 105)
axis.legend(loc="lower right")
axis.set_title("Right answers with the broken context and after the fix, per problem")
plt.tight_layout()
plt.show()

# ==========================================================================
# Step 7: Self-check
# ==========================================================================
print()
print("=" * 70)
print("✅ SELF-CHECK")
print("=" * 70)
bad_example = build_bad_context(example_question)
bad_text = json.dumps(bad_example["messages"])
print("✅ The bad context has all five problems: the wrong limit, both documents in full, 40 tools, no grounding instruction"
      if "Your daily limit is $1,000." in bad_text and TERMS_TEXT in bad_example["messages"][0]["content"]
      and OLD_FAQ_TEXT in bad_example["messages"][0]["content"] and len(bad_example["tools"]) == 40
      and GROUNDING_RULE not in bad_text
      else "❌ The bad context should hold RAW_TURNS unchecked, both raw documents in full, ALL_TOOLS, and the plain SYSTEM_PROMPT")
fixed_problems = []
for question in QUESTIONS:
    context = build_context(question["question"])
    text = json.dumps(context["messages"])
    system_text = context["messages"][0]["content"]
    right_title = next(section["title"] for section in POLICY_SECTIONS if section["number"] == question["section"])
    headings = re.findall(r"(?m)^## \d+\.\d+ ([^:\n]+): (.+)$", system_text)   # (label, title) of every section in the context
    tool_names = [tool["function"]["name"] for tool in context["tools"]]
    clean = ("$1,000" not in text and "$2.50" not in text and GROUNDING_RULE in system_text
             and len(headings) == 2 and all(label == "Personal accounts" for label, _ in headings)
             and right_title in [title.strip() for _, title in headings]
             and len(tool_names) == 3 and right_tool_for(question) in tool_names)
    if not clean:
        fixed_problems.append(question["id"])
print("✅ The fixed context has none of them, for all 20 questions: checked history, two matching personal sections, "
      "three matching tools, the grounding instruction" if not fixed_problems
      else f"❌ The fixed context still has a problem for: {', '.join(fixed_problems)}. Check each of the five fixes in build_context")
size_ratio = context_tokens(bad_example) / context_tokens(build_context(example_question))
print(f"✅ The fixed context is about {size_ratio:.0f} times smaller than the bad one" if size_ratio >= 20
      else f"❌ The fixed context should be at least 20 times smaller than the bad one (it is {size_ratio:.1f} times smaller)")
print("✅ Both runs have 25 answers, and the final table has 6 rows"
      if bad_run["total"] == 25 and fixed_run["total"] == 25 and len(final_table) == 6
      else f"❌ Expected 25 answers per run and 6 rows in the final table (it has {len(final_table)}: run Exercises 4 to 8 first)")
if fixed_run["right"] > bad_run["right"]:
    print(f"🤔 The bad context got {bad_run['right']} of 25 right, the fixed one {fixed_run['right']} of 25, "
          f"with about {bad_run['tokens_in'] // max(1, fixed_run['tokens_in'])} times fewer input tokens.")
else:
    print(f"🤔 The bad context got {bad_run['right']} of 25 right and the fixed one {fixed_run['right']} of 25: in your run "
          "the model coped with the bad context. It needed about "
          f"{bad_run['tokens_in'] // max(1, fixed_run['tokens_in'])} times the input tokens to do it, and nothing in that "
          "context stops the wrong limit or the old fee from coming back in the next run.")
print("🧠 Context engineering is deciding what the model reads: true, needed, current, and used. It is mostly ordinary code.")

# 🏁 Wrap-up: What You Did

| Exercise | What you did | Level |
|---|---|---|
| 1. Assemble the Context and Count the Tokens of Each Layer | Built the bot's context from four layers, saw where its tokens go, and measured the baseline on 20 questions | 🟢 |
| 2. Hold a Ten-Turn Conversation | Saw that every turn resends the whole history, and what that costs compared with ten separate calls | 🟢 |
| 3. Put the Current Time First, Then Last | Read the cached tokens of each call: nothing reused with the time on top, the stable part reused with the time at the end | 🟡 |
| 4. Poison the History, Then Check Replies | Counted the answers that repeat a wrong limit, and kept the wrong reply out of the history with a check in code | 🟡 |
| 5. Give the Bot the Full Terms and Conditions | Measured that sixty pages in every call cost about fifty times the input tokens of two selected sections, then selected the matching sections and asked for quotes that code checks | 🟡 |
| 6. Offer the Bot 40 Tools | Counted calls to a wrong tool, and passed only the tools that match the question | 🟡 |
| 7. Put an Old FAQ Page Next to the Policy | Counted two different answers to one question, then dated the sources and removed the outdated one with code | 🟡 |
| 8. Ask Five Questions With Unusual Answers | Saw what the model says from memory and that it uses the policy once it is there, and grounded the bot so that its answers come with a quote that code checks | 🟡 |
| 9. Put Every Fix Into `build_context()` | Built a clean context from messy material and compared it with a context that has all five problems | 🔴 |

### 🧾 The habit
When a bot gives a wrong answer, **print the context of that call** before you touch the prompt. Look at each layer and ask: is this true, is this needed, is this current, and did the model use it?

### 🔑 The sentences to keep
- *The model knows only what is in the context of this call, and you pay for every token of it, every time.*
- *A context fails in five ways: a wrong fact stays (poisoning), the right paragraph is lost (distraction), too many tools (confusion), two sources disagree (clash), or the model does not use what it was given (ignored context). Most of the fixes are a few lines of ordinary code that decide what the model gets to read.*